<a href="https://colab.research.google.com/github/srishtisushi/ETL_Pollution_Outcomes/blob/main/Final_srishti_rajeev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final Project: Transportation, Pollution, and Health Outcomes

## Background
The goal of my final project was to create an RDB that would allow end users to explore associations between levels of various pollutants (O2, NO2, PM2.5) health outcomes (asthma, stroke, COPD), and transportation patterns. I wanted users to be able to query this information by county or by the larger region that county belonged to.

*Note that counties are typically classified by a unique Federal Information Processing Standard or FIPS code, and a larger way to regionally group them is through looking at Core-Based Statistical Area or CBSA. Each state has its own FIPS code, and so does each county within that state. I will be using both FIPS and CBSA codes throughout this notebook/project.  


## Data Sources

There were several data sources I used for this project.

**CDC PLACES Data:** Has data on crude prevalence of various health outcomes for 3,000+ counties in 2022, along with the estimated populations and 18+ populations in those counties. I used the data portal accessible using the link below to query specifically for asthma, COPD, and stroke, and then stored the file on S3.

*   Original Data & Documentation: https://data.cdc.gov/500-Cities-Places/
*   My data file stored on S3: https://srajeevista322final.s3.us-east-2.amazonaws.com/PLACES__Local_Data_for_Better_Health__Census_Tract_Data_2024_release_20250506.csv

**EPA Data:** The EPA collects daily air quality measurements for various pollutants. I used their AQI service to query 2022 annual mean levels across thousands of sites for three main pollutants known to be associated with poorer health outcomes: NO2, ozone, and PM2.5 a.k.a particulate matter. I queried 3 separate times for each pollutant, converted the JSON files to CSV, and put them into my S3. This data has site ID, county FIPS codes, county, state, and cbsa name/cbsa codes. Note, however, that many of the cbsa codes/names are blank.

*   Original Data & Documentation : https://aqs.epa.gov/aqsweb/documents/data_api.html
*   Ozone file on S3: https://srajeevista322final.s3.us-east-2.amazonaws.com/ozone_by_site.csv
*   NO2 file on S3: https://srajeevista322final.s3.us-east-2.amazonaws.com/no2_by_site_updated.csv
*   PM2.5 file on S3: https://srajeevista322final.s3.us-east-2.amazonaws.com/pm25_by_site.csv


**NHTS Data:** This data, generated from thousands of surveys, shows the annual number of trips from one place to another, what the typical mode of transport between those two zones were, how long in distance the trip took, and whether it was for work or not. Note that the initial data describes locations using "zone id", but they also provide a lookup file they themselves made that you can use to convert all zone_ids into cbsa codes.

*   NHTS 2022 Data & Lookup Files downloaded from: https://nhts.ornl.gov/od/downloads
*   Data Documentation: https://nhts.ornl.gov/documentation
*   NHTS 2022 Zone Lookup File on S3: https://srajeevista322final.s3.us-east-2.amazonaws.com/NHTSLOOKUP.csv
*   NHTS 2022 Trip Data File on S3: https://srajeevista322final.s3.us-east-2.amazonaws.com/PASSENGER_DATA.csv



**CBSA to FIPS Crosswalk File**: I realized that since I had both CBSA and county FIPS in some of my files, I wanted to map the two. NHTS data was only in CBSA codes but my EPA-CDC data was primarily at the county level. I also knew after looking through my file that I was likely going to use county FIPS as a primary key & way to merge data for my RDB, so I decided to get this handy crosswalk file. This file, provided by the National Bureau of Economic Research, links each county FIPS to CBSA codes and provides some helpful additional city, state, and region information too:

*   Original Data & Documentation: https://www.nber.org/research/data/census-core-based-statistical-area-cbsa-federal-information-processing-series-fips-county-crosswalk
*   Crosswalk file on S3: https://srajeevista322final.s3.us-east-2.amazonaws.com/cbsa2fipsxw+(1).csv


# Extracting Data

Here's several things I aimed to explore:
*   Data columns, values, data type formats, consistency across datasets, particularly in CBSA/FIPS codes and classification
*   Renaming/dropping columns
*  Missing values and deciding how to handle them



In [ ]:
# Alright, let's start by importing packages.
import pandas as pd
import numpy as np
import datetime
import math

# Alright now to load the raw data from my S3 bucket.
# Quick design justification:
# I initially used the EPA API to extract EPA data, but converted the results to CSV and uploaded them to S3 to streamline access as suggested in Lecture 9 and keep all files in one place.
conversion_table = pd.read_csv('https://srajeevista322final.s3.us-east-2.amazonaws.com/cbsa2fipsxw+(1).csv')
epa_ozone = pd.read_csv('https://srajeevista322final.s3.us-east-2.amazonaws.com/ozone_by_site.csv')
epa_no2 = pd.read_csv('https://srajeevista322final.s3.us-east-2.amazonaws.com/no2_by_site_updated.csv')
epa_pm25 = pd.read_csv('https://srajeevista322final.s3.us-east-2.amazonaws.com/pm25_by_site.csv')
cdc_raw = pd.read_csv('https://srajeevista322final.s3.us-east-2.amazonaws.com/PLACES__Local_Data_for_Better_Health__Census_Tract_Data_2024_release_20250506.csv')
passenger_raw = pd.read_csv('https://srajeevista322final.s3.us-east-2.amazonaws.com/PASSENGER_DATA.csv')
nhtslookup = pd.read_csv('https://srajeevista322final.s3.us-east-2.amazonaws.com/NHTSLOOKUP.csv')

## Exploring CBSA-County Crosswalk Conversion Table

In [ ]:
# Ok let's start by exploring the conversion table.
conversion_table

,cbsacode,metropolitandivisioncode,csacode,cbsatitle,metropolitanmicropolitanstatis,metropolitandivisiontitle,csatitle,countycountyequivalent,statename,fipsstatecode,fipscountycode,centraloutlyingcounty
0,33860,NaN,388.0,"Montgomery, AL",Metropolitan Statistical Area,NaN,"Montgomery-Selma, AL",Autauga County,Alabama,1,1,Central
1,19300,NaN,380.0,"Daphne-Fairhope-Foley, AL",Metropolitan Statistical Area,NaN,"Mobile-Daphne-Fairhope, AL",Baldwin County,Alabama,1,3,Central
2,21640,NaN,NaN,"Eufaula, AL-GA",Micropolitan Statistical Area,NaN,NaN,Barbour County,Alabama,1,5,Central
3,13820,NaN,142.0,"Birmingham, AL",Metropolitan Statistical Area,NaN,"Birmingham-Cullman-Talladega, AL",Bibb County,Alabama,1,7,Outlying
4,13820,NaN,142.0,"Birmingham, AL",Metropolitan Statistical Area,NaN,"Birmingham-Cullman-Talladega, AL",Blount County,Alabama,1,9,Outlying
...,...,...,...,...,...,...,...,...,...,...,...,...
1910,41980,NaN,490.0,"San Juan-Bayamón-Caguas, PR",Metropolitan Statistical Area,NaN,"San Juan-Bayamón, PR",Vega Alta Municipio,Puerto Rico,72,143,Central
1911,41980,NaN,490.0,"San Juan-Bayamón-Caguas, PR",Metropolitan Statistical Area,NaN,"San Juan-Bayamón, PR",Vega Baja Municipio,Puerto Rico,72,145,Central
1912,38660,NaN,434.0,"Ponce, PR",Metropolitan Statistical Area,NaN,"Ponce-Coamo, PR",Villalba Municipio,Puerto Rico,72,149,Outlying
1913,41980,NaN,490.0,"San Juan-Bayamón-Caguas, PR",Metropolitan Statistical Area,NaN,"San Juan-Bayamón, PR",Yabucoa Municipio,Puerto Rico,72,151,Central


Ok so this will be useful for converting counties to CBSAs. Note: csa codes refer to an even bigger grouping of cbsas. If the CBSAs don't match with other tables going forward during merges, I want to keep those additional state/county descriptions to see if I can find out why. So I think it'll be handy to keep these for now. However, I don't think any of the metropolitan information or centraloutlingcounty columns are necessary since I won't be merging on those, and since I already have state/county descriptions to understand CBSA codes more in case something goes wrong during the merge, keeping metropolitan information is kind of redundant. Many of the data in those columns are either NaN or duplicates anyway.

In [ ]:
# I'm going to remove any columns relating to metropolitan areas and titles, since that isn't relevant in any of our other tables. However, I'm going to keep county, state, fips codes, cbsa code, and csa code.
# Like we did in a previous notebook, let's make a copy of the data and make sure our drop function actually works.
conversion_dropped = conversion_table.copy(deep=True)
conversion_dropped.drop(['metropolitandivisioncode', 'metropolitanmicropolitanstatis', 'metropolitandivisiontitle', 'centraloutlyingcounty'] ,axis='columns')

,cbsacode,csacode,cbsatitle,csatitle,countycountyequivalent,statename,fipsstatecode,fipscountycode
0,33860,388.0,"Montgomery, AL","Montgomery-Selma, AL",Autauga County,Alabama,1,1
1,19300,380.0,"Daphne-Fairhope-Foley, AL","Mobile-Daphne-Fairhope, AL",Baldwin County,Alabama,1,3
2,21640,NaN,"Eufaula, AL-GA",NaN,Barbour County,Alabama,1,5
3,13820,142.0,"Birmingham, AL","Birmingham-Cullman-Talladega, AL",Bibb County,Alabama,1,7
4,13820,142.0,"Birmingham, AL","Birmingham-Cullman-Talladega, AL",Blount County,Alabama,1,9
...,...,...,...,...,...,...,...,...
1910,41980,490.0,"San Juan-Bayamón-Caguas, PR","San Juan-Bayamón, PR",Vega Alta Municipio,Puerto Rico,72,143
1911,41980,490.0,"San Juan-Bayamón-Caguas, PR","San Juan-Bayamón, PR",Vega Baja Municipio,Puerto Rico,72,145
1912,38660,434.0,"Ponce, PR","Ponce-Coamo, PR",Villalba Municipio,Puerto Rico,72,149
1913,41980,490.0,"San Juan-Bayamón-Caguas, PR","San Juan-Bayamón, PR",Yabucoa Municipio,Puerto Rico,72,151


In [ ]:
# Ok since the command above worked as desired, I'm going to actually drop the columns.
conversion_dropped = conversion_dropped.drop(['metropolitandivisioncode', 'metropolitanmicropolitanstatis', 'metropolitandivisiontitle', 'centraloutlyingcounty'] ,axis='columns')
# I'm also going to change column names to keep them consistent across tables.
conversion_dropped = conversion_dropped.rename(columns={'countycountyequivalent':'county', 'statename':'state'})
# I want all cbsa codes to be reported as integers, and not floats (since there's nothing after the decimal). It seems like some are in float and others are in integers. So let's ensure best format:
conversion_dropped['cbsacode'] = pd.to_numeric(conversion_dropped['cbsacode'], errors='coerce').astype('Int64')
print(conversion_dropped.head())
print(conversion_dropped.dtypes)
# Note: NaN values in csa code are expected since not all CBSAs belong to a CSA grouping. Those rows are still valid. I probably won't use CSA going forward, but juuuust in case there's a concern during merging we'll keep it for now.

   cbsacode  csacode                  cbsatitle  \
0     33860    388.0             Montgomery, AL   
1     19300    380.0  Daphne-Fairhope-Foley, AL   
2     21640      NaN             Eufaula, AL-GA   
3     13820    142.0             Birmingham, AL   
4     13820    142.0             Birmingham, AL   

                           csatitle          county    state  fipsstatecode  \
0              Montgomery-Selma, AL  Autauga County  Alabama              1   
1        Mobile-Daphne-Fairhope, AL  Baldwin County  Alabama              1   
2                               NaN  Barbour County  Alabama              1   
3  Birmingham-Cullman-Talladega, AL     Bibb County  Alabama              1   
4  Birmingham-Cullman-Talladega, AL   Blount County  Alabama              1   

   fipscountycode  
0               1  
1               3  
2               5  
3               7  
4               9  
cbsacode            Int64
csacode           float64
cbsatitle          object
csatitle           o

In [ ]:
# So, I'll retain the csacode columns & NA values in case I need it for later, and I'll convert csacode to integer just in case.
conversion_dropped['csacode'] = conversion_dropped['csacode'].astype('Int64')
print(conversion_dropped.dtypes) # check if it actually converted
print(conversion_dropped.head())

cbsacode           Int64
csacode            Int64
cbsatitle         object
csatitle          object
county            object
state             object
fipsstatecode      int64
fipscountycode     int64
dtype: object
   cbsacode  csacode                  cbsatitle  \
0     33860      388             Montgomery, AL   
1     19300      380  Daphne-Fairhope-Foley, AL   
2     21640     <NA>             Eufaula, AL-GA   
3     13820      142             Birmingham, AL   
4     13820      142             Birmingham, AL   

                           csatitle          county    state  fipsstatecode  \
0              Montgomery-Selma, AL  Autauga County  Alabama              1   
1        Mobile-Daphne-Fairhope, AL  Baldwin County  Alabama              1   
2                               NaN  Barbour County  Alabama              1   
3  Birmingham-Cullman-Talladega, AL     Bibb County  Alabama              1   
4  Birmingham-Cullman-Talladega, AL   Blount County  Alabama              1   

   f

In [ ]:
# Okay let's make sure theres no duplicates:
print(conversion_dropped['cbsacode'].nunique())
print(conversion_dropped.shape)
conversion_dropped[['fipsstatecode', 'fipscountycode']].drop_duplicates().shape[0]
# Alright it looks like there's no duplicates in fips codes. It looks like there's 1915 unique counties (unique fips state/county code combos) that map to 936 unique CBSA codes. Good to know

935
(1915, 8)


1915

## Exploring EPA Data

In [ ]:
# I'm going to explore all three of my EPA dataframes. They should have similar formats.
epa_ozone

,State FIPS Code,State,County FIPS Code,County,CBSA Code,CBSA Name,Site ID,Daily Max 8-hour Ozone Concentration,Daily AQI Value,Units,POC
0,42,Pennsylvania,1,Adams,23900.0,"Gettysburg, PA",420010001,0.042152,39.418034,ppm,1.0
1,42,Pennsylvania,1,Adams,23900.0,"Gettysburg, PA",420019991,0.039752,37.225353,ppm,1.0
2,42,Pennsylvania,101,Philadelphia,37980.0,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",421010004,0.035149,33.146553,ppm,2.0
3,42,Pennsylvania,101,Philadelphia,37980.0,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",421010024,0.039518,38.563380,ppm,1.0
4,42,Pennsylvania,101,Philadelphia,37980.0,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",421010048,0.037885,36.618910,ppm,1.0
...,...,...,...,...,...,...,...,...,...,...,...
1232,46,South Dakota,29,Codington,47980.0,"Watertown, SD",460290002,0.039571,37.423077,ppm,3.0
1233,46,South Dakota,33,Custer,39660.0,"Rapid City, SD",460330132,0.044179,41.940170,ppm,3.0
1234,46,South Dakota,71,Jackson,NaN,NaN,460710001,0.042738,40.543663,ppm,3.0
1235,46,South Dakota,93,Meade,39660.0,"Rapid City, SD",460930001,0.041243,38.365920,ppm,3.0


In [ ]:
epa_no2

,State FIPS Code,State,County FIPS Code,County,CBSA Code,CBSA Name,Site ID,Daily Max 1-hour NO2 Concentration,Daily AQI Value,Units,POC
0,34,New Jersey,11,Cumberland,47220.0,"Vineland-Bridgeton, NJ",340110007,14.567135,13.306180,ppb,1.0
1,34,New Jersey,13,Essex,35620.0,"New York-Newark-Jersey City, NY-NJ-PA",340130003,28.234602,26.231940,ppb,1.0
2,34,New Jersey,17,Hudson,35620.0,"New York-Newark-Jersey City, NY-NJ-PA",340170006,28.726665,26.680555,ppb,1.0
3,34,New Jersey,17,Hudson,35620.0,"New York-Newark-Jersey City, NY-NJ-PA",340171002,30.001377,27.925620,ppb,1.0
4,34,New Jersey,23,Middlesex,35620.0,"New York-Newark-Jersey City, NY-NJ-PA",340230011,17.534796,16.117807,ppb,2.0
...,...,...,...,...,...,...,...,...,...,...,...
903,29,Missouri,95,Jackson,28140.0,"Kansas City, MO-KS",290950034,22.022190,20.334270,ppb,1.0
904,29,Missouri,95,Jackson,28140.0,"Kansas City, MO-KS",290950042,22.046518,20.381615,ppb,1.0
905,53,Washington,33,King,42660.0,"Seattle-Tacoma-Bellevue, WA",530330030,28.604002,26.568571,ppb,1.0
906,53,Washington,33,King,42660.0,"Seattle-Tacoma-Bellevue, WA",530330080,22.269588,20.586302,ppb,1.0


In [ ]:
epa_pm25

,State FIPS Code,State,County FIPS Code,County,Site ID,Daily Mean PM2.5 Concentration,Daily AQI Value,Units,POC
0,13.0,Georgia,115.0,Floyd,131150003.0,7.051685,36.058990,ug/m3 LC,3.000000
1,13.0,Georgia,121.0,Fulton,131210039.0,8.162295,41.672130,ug/m3 LC,1.000000
2,13.0,Georgia,121.0,Fulton,131210055.0,10.194328,49.274628,ug/m3 LC,3.000000
3,13.0,Georgia,121.0,Fulton,131210056.0,7.947486,40.840782,ug/m3 LC,2.324022
4,13.0,Georgia,127.0,Glynn,131270006.0,8.620026,42.927296,ug/m3 LC,11.285714
...,...,...,...,...,...,...,...,...,...
1302,46.0,South Dakota,29.0,Codington,460290002.0,6.874022,36.628490,ug/m3 LC,3.000000
1303,46.0,South Dakota,33.0,Custer,460330132.0,3.924786,20.679487,ug/m3 LC,2.500000
1304,46.0,South Dakota,65.0,Hughes,460650003.0,1.918232,11.143646,ug/m3 LC,3.000000
1305,46.0,South Dakota,71.0,Jackson,460710001.0,3.257025,17.444216,ug/m3 LC,2.495868


Okay several things. Each dataframe is a different size of rows. Both while downloading/reading documentation, and scanning through, I realized this is because not all states collect pollutant data for ALL three (some only collect on ozone, others no2, etc.) and even within states each individual site may not collect all three pollutant levels. So this is expected, and we'll make sure to keep this in mind when we later merge these into one big EPA table. It looks like the EPA ALSO provides CBSA codes/names for this, which is nice. However, these are only present in epa_no2 and epa_ozone data - CBSA stuff isn't in the epa_pm25 file.

 We might be able to cross-reference this with the crosswalk file to see if it's accurate. Also note that when I hit the calculator icon and scroll through I do see a lot of missing CBSA codes/names in both those tables. Some counties won't match to a CBSA code, as expected, but there's a possibility that the CBSA code just wasn't provided in the EPA data. We'll keep this in mind for later.

In [ ]:
# Ok I'm noticing several data formatting concerns when I scroll through the data files (hit calculator icon).
# First, site ID/state FIPS code has decimals in some tables, and even within tables aren't formatted consistently (some entries have decimals like 0.0, others don't).
# Let's ensure correct formatting across all three tables and across all of those columns:
for df in [epa_ozone, epa_no2]:
    df['Site ID'] = pd.to_numeric(df['Site ID'], errors='coerce').astype('Int64')
    df['State FIPS Code'] = pd.to_numeric(df['State FIPS Code'], errors='coerce').astype('Int64')
    df['County FIPS Code'] = pd.to_numeric(df['County FIPS Code'], errors='coerce').astype('Int64')
    df['CBSA Code'] = pd.to_numeric(df['CBSA Code'], errors='coerce').astype('Int64')

# Let's also do that for the PM_25 data (ran a separate loop because it didn't have CBSA codes/names.)
for df in [epa_pm25]:
    df['Site ID'] = pd.to_numeric(df['Site ID'], errors='coerce').astype('Int64')
    df['State FIPS Code'] = pd.to_numeric(df['State FIPS Code'], errors='coerce').astype('Int64')
    df['County FIPS Code'] = pd.to_numeric(df['County FIPS Code'], errors='coerce').astype('Int64')
print(epa_pm25.head())
print(epa_ozone.head())
print(epa_no2.head())
print(epa_pm25.dtypes)
print(epa_ozone.dtypes)
print(epa_no2.dtypes)

   State FIPS Code    State  County FIPS Code  County    Site ID  \
0               13  Georgia               115   Floyd  131150003   
1               13  Georgia               121  Fulton  131210039   
2               13  Georgia               121  Fulton  131210055   
3               13  Georgia               121  Fulton  131210056   
4               13  Georgia               127   Glynn  131270006   

   Daily Mean PM2.5 Concentration  Daily AQI Value     Units        POC  
0                        7.051685        36.058990  ug/m3 LC   3.000000  
1                        8.162295        41.672130  ug/m3 LC   1.000000  
2                       10.194328        49.274628  ug/m3 LC   3.000000  
3                        7.947486        40.840782  ug/m3 LC   2.324022  
4                        8.620026        42.927296  ug/m3 LC  11.285714  
   State FIPS Code         State  County FIPS Code        County  CBSA Code  \
0               42  Pennsylvania                 1         Adams    

In [ ]:
# POC just refers to how many devices were in the area. I kept this column initially because I wanted to ensure there WERE actual recordings for each entry.
# If this is always 1 or greater, I'm going to remove it since it's not zero and there's actually data collected
for df in [epa_pm25, epa_ozone, epa_no2]:
    print(df['POC'].min())
# Ok since it's always 1 or greater across all three dataframes, I'm going to drop it later.

1.0
1.0
1.0


In [ ]:
# Secondly, having an entire column of "Units" is also redundant
# I'll just rename the value column name to include units across all 3 tables and drop that column.
# But first let's check the units truly are consistent/redundant throughout:
for df in [epa_pm25, epa_ozone, epa_no2]:
    print(df['Units'].unique())

['ug/m3 LC' nan]
['ppm']
['ppb']


In [ ]:
#Uh oh. It looks like there's an nan value in epa_pm25. Let's check for NaN values across all columns of dataframes, actually:
for name, df in zip(['PM2.5', 'Ozone', 'NO2'], [epa_pm25, epa_ozone, epa_no2]):
    print(f"\n{name} missing:\n{df.isna().sum()}")


PM2.5 missing:
State FIPS Code                   1
State                             1
County FIPS Code                  1
County                            1
Site ID                           1
Daily Mean PM2.5 Concentration    1
Daily AQI Value                   1
Units                             1
POC                               1
dtype: int64

Ozone missing:
State FIPS Code                           0
State                                     0
County FIPS Code                          0
County                                    0
CBSA Code                               130
CBSA Name                               130
Site ID                                   0
Daily Max 8-hour Ozone Concentration      0
Daily AQI Value                           0
Units                                     0
POC                                       0
dtype: int64

NO2 missing:
State FIPS Code                        0
State                                  0
County FIPS Code                      

In [ ]:
# Since it was 1 NA entry in all columns for PM2.5 I'm guessing there's just a row of NaNs. Let's see if this is the case, and if so, if I can delete/remove that row.
epa_pm25[epa_pm25.isna().any(axis=1)]

,State FIPS Code,State,County FIPS Code,County,Site ID,Daily Mean PM2.5 Concentration,Daily AQI Value,Units,POC
524,<NA>,NaN,<NA>,NaN,<NA>,NaN,NaN,NaN,NaN


In [ ]:
# The problem seems to be line 524 (it's just a row of NAs, so I'm going to remove it).
epa_pm25 = epa_pm25.drop(index=524)

In [ ]:
# The NAs should be in CBSA code & name for both ozone and NO2, but let's scan those rows to make sure there's nothing else weird going on.
epa_no2[epa_no2.isna().any(axis=1)]

,State FIPS Code,State,County FIPS Code,County,CBSA Code,CBSA Name,Site ID,Daily Max 1-hour NO2 Concentration,Daily AQI Value,Units,POC
29,24,Maryland,23,Garrett,<NA>,NaN,240230002,2.763380,2.278873,ppb,4.0
76,56,Wyoming,19,Johnson,<NA>,NaN,560190004,1.349725,0.942308,ppb,1.0
78,56,Wyoming,23,Lincoln,<NA>,NaN,560230004,14.732958,13.470423,ppb,1.0
81,56,Wyoming,35,Sublette,<NA>,NaN,560350099,2.478134,1.979592,ppb,1.0
82,56,Wyoming,35,Sublette,<NA>,NaN,560350100,1.125627,0.637883,ppb,1.0
...,...,...,...,...,...,...,...,...,...,...,...
749,38,North Dakota,57,Mercer,<NA>,NaN,380570124,3.930556,3.833333,ppb,1.0
893,19,Iowa,177,Van Buren,<NA>,NaN,191770006,2.956686,2.468023,ppb,1.0
894,20,Kansas,133,Neosho,<NA>,NaN,201330003,6.958537,6.278049,ppb,1.0
897,20,Kansas,195,Trego,<NA>,NaN,201950001,3.054985,2.586103,ppb,1.0


In [ ]:
epa_ozone[epa_ozone.isna().any(axis=1)]

,State FIPS Code,State,County FIPS Code,County,CBSA Code,CBSA Name,Site ID,Daily Max 8-hour Ozone Concentration,Daily AQI Value,Units,POC
8,42,Pennsylvania,117,Tioga,<NA>,NaN,421174000,0.040408,37.745834,ppm,1.000000
34,42,Pennsylvania,59,Greene,<NA>,NaN,420590002,0.042918,40.936073,ppm,1.000000
78,56,Wyoming,19,Johnson,<NA>,NaN,560190004,0.042542,39.593220,ppm,1.000000
80,56,Wyoming,23,Lincoln,<NA>,NaN,560230004,0.045726,42.768570,ppm,1.000000
83,56,Wyoming,3,Big Horn,<NA>,NaN,560030002,0.040147,37.229916,ppm,1.000000
...,...,...,...,...,...,...,...,...,...,...,...
1209,29,Missouri,137,Monroe,<NA>,NaN,291370001,0.036112,33.430790,ppm,1.494350
1210,29,Missouri,157,Perry,<NA>,NaN,291570001,0.044354,42.896843,ppm,1.507368
1213,29,Missouri,186,Sainte Genevieve,<NA>,NaN,291860005,0.043559,41.668068,ppm,1.489496
1220,29,Missouri,39,Cedar,<NA>,NaN,290390001,0.044898,43.217213,ppm,1.500000


In [ ]:
# Okay so it looks like there are genuinely just CBSA codes/names missing for ozone and NO2. For now, I'll mark these explicitly like "UNKNOWN" so that later when I transform the data, I know to check for these.
for df in [epa_ozone, epa_no2]:
    df['CBSA Name'] = df['CBSA Name'].fillna('UNKNOWN')

In [ ]:
# Ok since our units column data are truly consistent/redundant (it was just that one nan value for pm25), I'm going to rename the columns & drop units & POC
epa_pm25 = epa_pm25.rename(columns={'Daily Mean PM2.5 Concentration': 'PM2.5 Concentration (ug/m3)'})
epa_pm25 = epa_pm25.drop(columns=['Units', 'POC'])
epa_ozone = epa_ozone.rename(columns={'Daily Max 8-hour Ozone Concentration': 'Ozone Concentration (ppm)'})
epa_ozone = epa_ozone.drop(columns=['Units', 'POC'])
epa_no2 = epa_no2.rename(columns={'Daily Max 1-hour NO2 Concentration': 'NO2 Concentration (ppb)'})
epa_no2 = epa_no2.drop(columns=['Units', 'POC'])

#Let's also rename the columns to be consistent with the earlier conversion table naming:
epa_pm25 = epa_pm25.rename(columns={
    'State FIPS Code': 'fipsstatecode',
    'County FIPS Code': 'fipscountycode',
    'State': 'state',
    'County': 'county'})
epa_no2 = epa_no2.rename(columns={
    'State FIPS Code': 'fipsstatecode',
    'County FIPS Code': 'fipscountycode',
    'State': 'state',
    'County': 'county',
    'CBSA Code': 'cbsacode',
    'CBSA Name': 'cbsaname',})
epa_ozone = epa_ozone.rename(columns={
    'State FIPS Code': 'fipsstatecode',
    'County FIPS Code': 'fipscountycode',
    'State': 'state',
    'CBSA Code': 'cbsacode',
    'CBSA Name': 'cbsaname',
    'County': 'county'})
print(epa_pm25.head())
print(epa_ozone.head())
print(epa_no2.head())

#Ok all column names are consistent now.

   fipsstatecode    state  fipscountycode  county    Site ID  \
0             13  Georgia             115   Floyd  131150003   
1             13  Georgia             121  Fulton  131210039   
2             13  Georgia             121  Fulton  131210055   
3             13  Georgia             121  Fulton  131210056   
4             13  Georgia             127   Glynn  131270006   

   PM2.5 Concentration (ug/m3)  Daily AQI Value  
0                     7.051685        36.058990  
1                     8.162295        41.672130  
2                    10.194328        49.274628  
3                     7.947486        40.840782  
4                     8.620026        42.927296  
   fipsstatecode         state  fipscountycode        county  cbsacode  \
0             42  Pennsylvania               1         Adams     23900   
1             42  Pennsylvania               1         Adams     23900   
2             42  Pennsylvania             101  Philadelphia     37980   
3             42  P

In [ ]:
# Let's double check that there's no duplicate rows. Since each site ID is unique, the length of each table should be equal to the number of unique site IDs.
print(len(epa_pm25) == epa_pm25['Site ID'].nunique())
print(len(epa_ozone) == epa_ozone['Site ID'].nunique())
print(len(epa_no2) == epa_no2['Site ID'].nunique())

True
True
False


In [ ]:
# Uh oh. That printed false for NO2. Let's check out what's going on by seeing which IDs are showing up, and how many times:
duplicates = epa_no2.groupby(['Site ID', 'state']).size().loc[lambda x: x > 1].reset_index(name='count')
duplicates

,Site ID,state,count
0,10730023,Alabama,2
1,10732059,Alabama,2
2,40130019,Arizona,2
3,40133002,Arizona,2
4,40134011,Arizona,2
...,...,...,...
449,560351002,Wyoming,2
450,560370200,Wyoming,2
451,560370300,Wyoming,2
452,560391013,Wyoming,2


In [ ]:
# Ok every site ID that has a count greater than 1 is 2, suggesting that it's an issue of duplicate rows.
# I'm going to copy the site ID from the duplicate table that just printed out and search for it in the previous No2 table.
epa_no2.head()
# When I filter for these site IDs (hit the calculator looking button), I do in fact see exact duplicate rows for the several codes I tried doing this with.

,fipsstatecode,state,fipscountycode,county,cbsacode,cbsaname,Site ID,NO2 Concentration (ppb),Daily AQI Value
0,34,New Jersey,11,Cumberland,47220,"Vineland-Bridgeton, NJ",340110007,14.567135,13.306180
1,34,New Jersey,13,Essex,35620,"New York-Newark-Jersey City, NY-NJ-PA",340130003,28.234602,26.231940
2,34,New Jersey,17,Hudson,35620,"New York-Newark-Jersey City, NY-NJ-PA",340170006,28.726665,26.680555
3,34,New Jersey,17,Hudson,35620,"New York-Newark-Jersey City, NY-NJ-PA",340171002,30.001377,27.925620
4,34,New Jersey,23,Middlesex,35620,"New York-Newark-Jersey City, NY-NJ-PA",340230011,17.534796,16.117807


In [ ]:
# Let's see if all site IDs that pop up multiple times are actually just due to duplicate rows:
duplicate_siteids = epa_no2[epa_no2['Site ID'].duplicated(keep=False)]
exact_dupes = duplicate_siteids.duplicated(keep=False).all()
print(exact_dupes)

True


In [ ]:
# Okay the "true" that just printed suggests that they're all just duplicate rows I need to remove.
epa_no2 = epa_no2.drop_duplicates()
epa_no2 = epa_no2.reset_index(drop=True)

# Let's check to see that was truly the error. If it was and this prints true, the length should be equal to the number of unique site IDs:
print(len(epa_no2) == epa_no2['Site ID'].nunique())

#YAY! That printed true.

True


In [ ]:
# Note: I think the reason for this, looking back, is because I accidentally merged the AQI data to -> csvs twice for NO2.
print(epa_no2['state'].nunique()) # Ok so I'm just using this to check there are actually 49 unique states as the documentation says there's supposed to be. We're good to proceed.
print(epa_ozone['state'].nunique()) # Also using these two commands I'm just using to show that there are some states that didn't collect data for certain pollutants (note data includes puerto rico/virgin islands, etc.). This difference is expected.
print(epa_pm25['state'].nunique())

49
52
53


In [ ]:
# Note: while I was scanning through the data and reading documentation I noticed that some sites didn't collect data on ALL three pollutants, just one or the other.
print((~epa_pm25['Site ID'].isin(epa_no2['Site ID'])).any())
print((~epa_pm25['Site ID'].isin(epa_ozone['Site ID'])).any())
print((~epa_ozone['Site ID'].isin(epa_no2['Site ID'])).any())
# Running the lines of code above quickly (or just examining the length of each dataframe since each row correpsonds to a unique site ID) verifies this observation, just showing that there are some sites are NOT present in other dataframes.
# Note: this would later influence how I'd aggregate all of the EPA data (it'd be by county FIPS rather than site ID), and then merging average values taken for various pollutants at various sites for that respective FIPS code.

True
True
True


## Exploring NHTS Transportation Data

In [ ]:
passenger_raw.head()

,year,origin_zone_id,destination_zone_id,origin_zone_name,destination_zone_name,origin_state,destination_state,annual_total_trips,mode_air,mode_rail,...,vehicle_gt300mi_pct,atf_0_10mi_pct,atf_10_25mi_pct,atf_25_50mi_pct,atf_50_75mi_pct,atf_75_100mi_pct,atf_100_150mi_pct,atf_150_300mi_pct,atf_gt300mi_pct,flag
0,2022,10180_TX,10180_TX,"Abilene, TX","Abilene, TX",TX,TX,179452461,0,0,...,0.0,0.991902,0.006386,0.001712,0.0,0.0,0.0,0.0,0.0,0
1,2022,10180_TX,10420_OH,"Abilene, TX","Akron, OH",TX,OH,71,0,0,...,1.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0
2,2022,10180_TX,10500_GA,"Abilene, TX","Albany, GA",TX,GA,105,0,0,...,1.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0
3,2022,10180_TX,10540_OR,"Abilene, TX","Albany, OR",TX,OR,0,0,0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,1
4,2022,10180_TX,10580_NY,"Abilene, TX","Albany-Schenectady-Troy, NY",TX,NY,37,0,0,...,1.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0


In [ ]:
# Alright, that's WAY too many columns to keep. Rather than specify the 100+ columns I want to drop, I figure it's more efficient to specify the few columns I want to keep.
#Let's start by printing out the columns:
print(passenger_raw.columns.to_list())

['year', 'origin_zone_id', 'destination_zone_id', 'origin_zone_name', 'destination_zone_name', 'origin_state', 'destination_state', 'annual_total_trips', 'mode_air', 'mode_rail', 'mode_vehicle', 'mode_atf', 'purpose_work', 'purpose_nonwork', 'distance_0_10', 'distance_10_25', 'distance_25_50', 'distance_50_75', 'distance_75_100', 'distance_100_150', 'distance_150_300', 'distance_gt300', 'work_0_10mi', 'work_10_25mi', 'work_25_50mi', 'work_50_75mi', 'work_75_100mi', 'work_100_150mi', 'work_150_300mi', 'work_gt300mi', 'nonwork_0_10mi', 'nonwork_10_25mi', 'nonwork_25_50mi', 'nonwork_50_75mi', 'nonwork_75_100mi', 'nonwork_100_150mi', 'nonwork_150_300mi', 'nonwork_gt300mi', 'air_0_10mi', 'air_10_25mi', 'air_25_50mi', 'air_50_75mi', 'air_75_100mi', 'air_100_150mi', 'air_150_300mi', 'air_gt300mi', 'rail_0_10mi', 'rail_10_25mi', 'rail_25_50mi', 'rail_50_75mi', 'rail_75_100mi', 'rail_100_150mi', 'rail_150_300mi', 'rail_gt300mi', 'vehicle_0_10mi', 'vehicle_10_25mi', 'vehicle_25_50mi', 'vehicle_5

A bunch of these I don't need to keep. Anything with 'pcts' for example are just percent conversions of raw data. Keep in mind the columns are showing the NUMBER OF TRIPS that belong to that category. Since I plan to later aggregate by zip codes, I want to keep the raw number of trips rather than percent so I can re-calculate percentages. So I'll drop the percent columns. I also know that during transformation I'll re-categorize trips in a given zone by distance, so I'll keep the distance columns for now. Even though it's a lot, I'll later combine the 7-8 distance columns into just two (short and long). I also want to later see whether transportation in a given region is primarily for work or not later, so I'll keep work/nonwork columns in for now.

Also! The NHTS does flag data if it was suppressed or there was something else concerning going on. Scanning through the data earlier it seems like many of the rows with a flag of 1 just had zeroes in other columns (which might reveal limitations of the survey data, not ACTUAL trends in transportation, because again, this data was aggregated BASED off of household responses, so there might just not have been any respondents in that given zone). I'm going to keep this column for now so I can quickly remove rows that they flagged, but will do additional quality checks after.

Alright! Let's go ahead and pick/drop columns.

In [ ]:
columns_to_keep = ['origin_zone_id', 'destination_zone_id', 'origin_zone_name', 'destination_zone_name', 'origin_state', 'destination_state', 'annual_total_trips', 'mode_air', 'mode_rail', 'mode_vehicle', 'mode_atf', 'purpose_work', 'purpose_nonwork', 'distance_0_10', 'distance_10_25', 'distance_25_50', 'distance_50_75', 'distance_75_100', 'distance_100_150', 'distance_150_300', 'distance_gt300','flag']
passenger_cleaned = passenger_raw[columns_to_keep]
passenger_cleaned

,origin_zone_id,destination_zone_id,origin_zone_name,destination_zone_name,origin_state,destination_state,annual_total_trips,mode_air,mode_rail,mode_vehicle,...,purpose_nonwork,distance_0_10,distance_10_25,distance_25_50,distance_50_75,distance_75_100,distance_100_150,distance_150_300,distance_gt300,flag
0,10180_TX,10180_TX,"Abilene, TX","Abilene, TX",TX,TX,179452461,0,0,163715930,...,124757010,154451306,20007548,4322775,652433,18119,280,0,0,0
1,10180_TX,10420_OH,"Abilene, TX","Akron, OH",TX,OH,71,0,0,71,...,40,0,0,0,0,0,0,0,71,0
2,10180_TX,10500_GA,"Abilene, TX","Albany, GA",TX,GA,105,0,0,105,...,19,0,0,0,0,0,0,0,105,0
3,10180_TX,10540_OR,"Abilene, TX","Albany, OR",TX,OR,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,10180_TX,10580_NY,"Abilene, TX","Albany-Schenectady-Troy, NY",TX,NY,37,0,0,37,...,26,0,0,0,0,0,0,0,37,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
339884,RWY3_WY,RWV2_WV,WY-NonMSA areas (SW),WV-NonMSA areas (NW),WY,WV,72,0,0,72,...,18,0,0,0,0,0,0,0,72,0
339885,RWY3_WY,RWV3_WV,WY-NonMSA areas (SW),WV-NonMSA areas (S),WY,WV,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
339886,RWY3_WY,RWY1_WY,WY-NonMSA areas (SW),WY-NonMSA areas (E),WY,WY,308418,0,0,306236,...,166867,5625,16569,44007,67017,8236,36348,86593,44023,0
339887,RWY3_WY,RWY2_WY,WY-NonMSA areas (SW),WY-NonMSA areas (NW),WY,WY,462361,1180,0,458665,...,271573,648,5892,188651,106781,47384,50454,57798,4753,0


In [ ]:
#Alright, it looks like the raw data has 339889 rows. Let's remove flagged rows, and see how many rows are left.
passenger_cleaned = passenger_cleaned[passenger_cleaned['flag'] != 1]
print(passenger_cleaned.shape[0])
# Okay, 294841 rows is still quite sufficient.

294841


## Exploring CDC Data

In [ ]:
cdc_raw
# Okay. Seeing a consideration - the CDC reports not state fips and county fips as individual columns, but as a merged column.
# I actually like this format. It's more efficient and saves column/data space. I'll just merge the state/county fips columns in the other tables to generate something like this for unique merging based on the whole CountyFIPS.

,StateAbbr,StateDesc,CountyName,CountyFIPS,LocationName,Measure,Data_Value_Type,Data_Value,Data_Value_Footnote,Low_Confidence_Limit,High_Confidence_Limit,TotalPopulation,TotalPop18plus,Geolocation,LocationID,MeasureId,DataValueTypeID
0,FL,Florida,Miami-Dade,12086,12086010025,Stroke among adults,Crude prevalence,5.6,NaN,5.1,6.1,3992,3091,POINT (-80.254741 25.9664383),12086010025,STROKE,CrdPrv
1,FL,Florida,Miami-Dade,12086,12086010609,Stroke among adults,Crude prevalence,5.3,NaN,4.9,5.7,5636,4521,POINT (-80.3610726 25.5703151),12086010609,STROKE,CrdPrv
2,FL,Florida,Miami-Dade,12086,12086005701,Stroke among adults,Crude prevalence,4.4,NaN,4.0,4.8,5947,5173,POINT (-80.2967531 25.7732667),12086005701,STROKE,CrdPrv
3,FL,Florida,Miami-Dade,12086,12086005706,Stroke among adults,Crude prevalence,3.8,NaN,3.4,4.2,4213,3674,POINT (-80.2828435 25.7788869),12086005706,STROKE,CrdPrv
4,FL,Florida,Miami-Dade,12086,12086006105,Stroke among adults,Crude prevalence,3.3,NaN,3.0,3.6,2260,1796,POINT (-80.2838417 25.757788),12086006105,STROKE,CrdPrv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
250561,FL,Florida,Manatee,12081,12081002003,Current asthma among adults,Crude prevalence,9.4,NaN,8.3,10.5,6786,5643,POINT (-82.5036301 27.504205),12081002003,CASTHMA,CrdPrv
250562,FL,Florida,Manatee,12081,12081000310,Stroke among adults,Crude prevalence,3.8,NaN,3.4,4.1,4669,3458,POINT (-82.5808243 27.4541015),12081000310,STROKE,CrdPrv
250563,FL,Florida,Manatee,12081,12081000808,Stroke among adults,Crude prevalence,4.2,NaN,3.8,4.6,6199,5585,POINT (-82.5017383 27.4134252),12081000808,STROKE,CrdPrv
250564,FL,Florida,Leon,12073,12073000303,Current asthma among adults,Crude prevalence,12.4,NaN,11.0,14.0,4153,3049,POINT (-84.2397063 30.4153823),12073000303,CASTHMA,CrdPrv


In [ ]:
# A bunch of the columns aren't really necessary (like DataValueType, Geolocation, etc.) and are redundant. I'm going to remove columns I don't want, including the limits since I'm going to aggregate by CountyFIPS which will throw those off
cdc_data = cdc_raw.drop(columns=['MeasureId', 'DataValueTypeID', 'Geolocation', 'LocationName', 'Data_Value_Type', 'Data_Value_Footnote', 'StateAbbr', 'Low_Confidence_Limit', 'High_Confidence_Limit'])
# And as a later note: we'll calculate the age-adjusted prevalence, since they provide a population of just 18+ and the total popuation, and we'll add as a column.
#I'll keep locationId since I'll be aggregating all rows by countyfips and in case something goes wrong during transform it might be helpful to have this as an additional identifier.
#Renaming columns for consistent formatting...
cdc_data = cdc_data.rename(columns={'Data_Value': 'data_value_crude', 'StateDesc': 'state', 'CountyName': 'county'})

# And let's check that worked!
print(cdc_data.dtypes)
print(cdc_data.shape)
cdc_data.head()
#Lucky for us, CountyFIPS and the population data are already in integer formats, so I'll leave their datatypes as they are.

state                object
county               object
CountyFIPS            int64
Measure              object
data_value_crude    float64
TotalPopulation       int64
TotalPop18plus        int64
LocationID            int64
dtype: object
(250566, 8)


,state,county,CountyFIPS,Measure,data_value_crude,TotalPopulation,TotalPop18plus,LocationID
0,Florida,Miami-Dade,12086,Stroke among adults,5.6,3992,3091,12086010025
1,Florida,Miami-Dade,12086,Stroke among adults,5.3,5636,4521,12086010609
2,Florida,Miami-Dade,12086,Stroke among adults,4.4,5947,5173,12086005701
3,Florida,Miami-Dade,12086,Stroke among adults,3.8,4213,3674,12086005706
4,Florida,Miami-Dade,12086,Stroke among adults,3.3,2260,1796,12086006105


## Exploring the NHTS Lookup File

In [ ]:
nhtslookup

,index,zone_id,zone_name,states,CBSAFP2
0,1,10180_TX,"Abilene, TX",TX,10180
1,2,10420_OH,"Akron, OH",OH,10420
2,3,10500_GA,"Albany, GA",GA,10500
3,4,10540_OR,"Albany, OR",OR,10540
4,5,10580_NY,"Albany-Schenectady-Troy, NY",NY,10580
...,...,...,...,...,...
578,579,RWV2_WV,WV-NonMSA areas (NW),WV,RWV2
579,580,RWV3_WV,WV-NonMSA areas (S),WV,RWV3
580,581,RWY1_WY,WY-NonMSA areas (E),WY,RWY1
581,582,RWY2_WY,WY-NonMSA areas (NW),WY,RWY2


Okay it looks like this is a relatively small and simple file. The NHTS transportation data has zones, so this will be useful in converting those to CBSAs. For now I think I'm just going to rename the CBSAFP2 to CBSA Code. I'll keep the zone_name/states just in case none of the zone_id's align during transform. Note: the lookup table includes rural CBSAcodes that the NHTS seems to have CUSTOMLY made for rural areas it surveyed, which is why its datatype is object as opposed to integer as seen below. I will most likely be dropping these rural areas both from this lookup & NHTS data since they had so much missing data and since CBSAs in our crosswalk/EPA data are the standard and there's no rural areas in any of those.

In [ ]:
# Renaming & examining datatypes
nhtslookup = nhtslookup.rename(columns={'CBSAFP2': 'cbsacode'})
print(nhtslookup.head())
print(nhtslookup.dtypes)

   index   zone_id                    zone_name states cbsacode
0      1  10180_TX                  Abilene, TX     TX    10180
1      2  10420_OH                    Akron, OH     OH    10420
2      3  10500_GA                   Albany, GA     GA    10500
3      4  10540_OR                   Albany, OR     OR    10540
4      5  10580_NY  Albany-Schenectady-Troy, NY     NY    10580
index         int64
zone_id      object
zone_name    object
states       object
cbsacode     object
dtype: object


# Transforming the Data

Finally, I'm going to transform. Here are my main goals with transforming:

*   Aggregating all data for a particular CBSA or CountyFIPS code, and turning columns into rows/vice versa if necessary
*   I liked the CDC format of CountyFIPS, so I want to make it consistent across all tables reporting countyFIPS.
*   Merging all three of my EPA tables into one
*   Creating a CBSA lookup that uses values from the crosswalk, but also from the EPA in case there's missing ones I didn't catch.

***Transforming CDC Data***

*   I'll need to add an "age-adjusted" rate for each countyfips
*   I also want to aggregate into averages for EACH county FIPS code, but use the SUM of total population/population above 18 for each of the location IDs that make up the FIPS code.





In [ ]:
# Alright starting with the easiest thing to do, adding adjusted prevalence - keep in mind our crude value is the prevalence amongst ADULTS, so it's nice to see it adjusted for age. This is a standard formula from epidemiology practice.
cdc_data['adj_prev'] = ((cdc_data['data_value_crude'] / 100) * cdc_data['TotalPop18plus'] / cdc_data['TotalPopulation']) * 100
cdc_data.head()
# Checking to see our new column got added - it did.

,state,county,CountyFIPS,Measure,data_value_crude,TotalPopulation,TotalPop18plus,LocationID,adj_prev
0,Florida,Miami-Dade,12086,Stroke among adults,5.6,3992,3091,12086010025,4.336072
1,Florida,Miami-Dade,12086,Stroke among adults,5.3,5636,4521,12086010609,4.251473
2,Florida,Miami-Dade,12086,Stroke among adults,4.4,5947,5173,12086005701,3.827342
3,Florida,Miami-Dade,12086,Stroke among adults,3.8,4213,3674,12086005706,3.313838
4,Florida,Miami-Dade,12086,Stroke among adults,3.3,2260,1796,12086006105,2.622478


In [ ]:
# Next, I wanted to group data by county, then aggregate so that each county was its own row.
# I'm choosing to take the average crude & adjusted prevalence values across census tracts and also sum total and 18+ populations to get county-level totals.
county_measure = (cdc_data.groupby(['CountyFIPS', 'Measure'], as_index=False).agg(crude_prev = ('data_value_crude', 'mean'), adj_prev = ('adj_prev', 'mean'), total_population = ('TotalPopulation', 'sum'),total_pop_18plus = ('TotalPop18plus', 'sum')))

In [ ]:
# Now I want to pivot my data so that here's only one row per county, and each health condition is a SEPARATE column (since they were previously rows)
county_prev_wide = (
    county_measure
    .pivot(index='CountyFIPS',
           columns='Measure',
           values=['crude_prev','adj_prev']))

In [ ]:
# I had to do this in order to ensure my column names generated by the pivot were consistent, and that they were all lowercase.
county_prev_wide.columns = [
    f"{measure.lower().replace(' ','_')}_{stat}"
    for stat, measure in county_prev_wide.columns]
county_prev_wide = county_prev_wide.reset_index() # doing this to ensure countyFIPS becomes a column & i reset the index.

In [ ]:
# This pulls population columns from the earlier table; I'm going to drop duplicates since there's only one popuation row per county since it's repeated across measures.
county_pops = (county_measure[['CountyFIPS','total_population','total_pop_18plus']].drop_duplicates())

In [ ]:
# Alright now I'm going to add BACK state and county name using the ORIGINAL CDC data, so that I can query by name in the future. Also when I'm cross-checking with other tables I want to be able to see state/county info.
county_summary = county_pops.merge(county_prev_wide, on='CountyFIPS')
county_summary = county_summary.merge(cdc_data[['CountyFIPS','state','county']].drop_duplicates(subset='CountyFIPS'), on='CountyFIPS', how='left')
state_to_fips = dict(zip(conversion_dropped['state'], conversion_dropped['fipsstatecode']))
print(county_summary.head())
print(county_summary.shape)
# Okay county_summary looks good, and has 3143 unique rows for 3143 counties counties as expected by CDC data documentation.

   CountyFIPS  total_population  total_pop_18plus  \
0        1001             58805             44523   
1        1003            231767            182471   
2        1005             25223             20134   
3        1007             22293             17533   
4        1009             59134             45403   

   chronic_obstructive_pulmonary_disease_among_adults_crude_prev  \
0                                           8.382353               
1                                           8.716279               
2                                          11.800000               
3                                          10.825000               
4                                          10.362500               

   current_asthma_among_adults_crude_prev  stroke_among_adults_crude_prev  \
0                               10.447059                        3.911765   
1                               10.025581                        4.002326   
2                               11.455556

## Adjusting CountyFIPS Format & Adding CBSA Codes
Now that my CDC data is fixed, and I like the way its CountyFIPS is formatted, I'm briefly going to reformat other tables so that they are the same way, and then merge all CBSA codes to all the tables before proceeding. Again, the other datasets split the CountyFIPS into state and county FIPS code, so I need to fix that and create just one unified CountyFIPS column.

In [ ]:
# create CountyFIPS for conversion table
# This line creates unified 5 digit CountyFIPS by combining fipsstatecode, while the .zfill ensures leading zeroes are retained so that the total fips code length comes out to 5
# ^ For example, Autaga County in Alabama is 01001, its state code is 1 and its county code is also 1. so to ensure that it comes out to a unique 5 digit fips I'll fill in the blank spaces with zeroes.
# I'll be doing this for all tables that have state fips code & county fips code as separate columns.

conversion_dropped['CountyFIPS'] = (
    conversion_dropped['fipsstatecode'].astype(int).astype(str).str.zfill(2) +
    conversion_dropped['fipscountycode'].astype(int).astype(str).str.zfill(3)).astype(int)

epa_pm25['CountyFIPS'] = (
    epa_pm25['fipsstatecode'].astype('Int64').astype(str).str.zfill(2) +
    epa_pm25['fipscountycode'].astype('Int64').astype(str).str.zfill(3)).astype(int)

epa_ozone['CountyFIPS'] = (
    epa_ozone['fipsstatecode'].astype('Int64').astype(str).str.zfill(2) +
    epa_ozone['fipscountycode'].astype('Int64').astype(str).str.zfill(3)).astype(int)

epa_no2['CountyFIPS'] = (
    epa_no2['fipsstatecode'].astype('Int64').astype(str).str.zfill(2) +
    epa_no2['fipscountycode'].astype('Int64').astype(str).str.zfill(3)).astype(int)

In [ ]:
# Now we just need to add CBSA codes to all of these tables based off the CountyFIPS to cbsacode conversion we get from our conversion file.
cbsa = conversion_dropped[['CountyFIPS', 'cbsacode']]
# Merge into each EPA dataset and county summary
epa_pm25 = epa_pm25.merge(cbsa, on='CountyFIPS', how='left')
epa_no2 = epa_no2.merge(cbsa, on='CountyFIPS', how='left')
epa_ozone = epa_ozone.merge(cbsa, on='CountyFIPS', how='left')
county_summary = county_summary.merge(cbsa, on='CountyFIPS', how='left')

Note: Yes, I know that two of our EPA tables already have cbsacode as a column, so this will create cbsacode_x (the original) and cbsacode_y (the code provided in the conversion file). I'd like to see whether there is discrepancies between the two (ensure they match if they are both present), and in places where there's gaps in the EPA data, I want to see if there's crosswalk data that can fill it in. Likewise, I also want to see if there's gaps in our crosswalk data that EPA can fill in.

In [ ]:
epa_ozone.head()

,fipsstatecode,state,fipscountycode,county,cbsacode_x,cbsaname,Site ID,Ozone Concentration (ppm),Daily AQI Value,CountyFIPS,cbsacode_y
0,42,Pennsylvania,1,Adams,23900,"Gettysburg, PA",420010001,0.042152,39.418034,42001,23900
1,42,Pennsylvania,1,Adams,23900,"Gettysburg, PA",420019991,0.039752,37.225353,42001,23900
2,42,Pennsylvania,101,Philadelphia,37980,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",421010004,0.035149,33.146553,42101,37980
3,42,Pennsylvania,101,Philadelphia,37980,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",421010024,0.039518,38.563380,42101,37980
4,42,Pennsylvania,101,Philadelphia,37980,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",421010048,0.037885,36.618910,42101,37980


In [ ]:
# Alright let's see where all the gaps in the data are. Starting out with the EPA data: were there any values in the EPA's CBSAs that either don't match up with the CBSAs we pulled?
# For ozone:
epa_ozone[
    (epa_ozone['cbsacode_x'].notna()) &
    (epa_ozone['cbsacode_y'].notna()) &
    (epa_ozone['cbsacode_x'] != epa_ozone['cbsacode_y'])]

# Kay it does look like there's 41 mismatches. It looks like these are mainly due to slight differences in official/Bureau vs EPA classifications. Hmm...

,fipsstatecode,state,fipscountycode,county,cbsacode_x,cbsaname,Site ID,Ozone Concentration (ppm),Daily AQI Value,CountyFIPS,cbsacode_y
44,42,Pennsylvania,73,Lawrence,35260,"New Castle, PA",420730015,0.040636,38.698350,42073,38300
49,42,Pennsylvania,85,Mercer,49660,"Youngstown-Warren-Boardman, OH-PA",420850100,0.042865,41.118850,42085,25850
50,42,Pennsylvania,85,Mercer,49660,"Youngstown-Warren-Boardman, OH-PA",420859991,0.039011,36.827194,42085,25850
133,37,North Carolina,87,Haywood,11700,"Asheville, NC",370870008,0.041273,38.768597,37087,48200
134,37,North Carolina,87,Haywood,11700,"Asheville, NC",370870035,0.044307,42.004097,37087,48200
135,37,North Carolina,87,Haywood,11700,"Asheville, NC",370870036,0.045798,43.836136,37087,48200
154,55,Wisconsin,59,Kenosha,16980,"Chicago-Naperville-Elgin, IL-IN-WI",550590019,0.044232,44.000000,55059,28450
155,55,Wisconsin,59,Kenosha,16980,"Chicago-Naperville-Elgin, IL-IN-WI",550590025,0.042138,40.814228,55059,28450
203,24,Maryland,9,Calvert,47900,"Washington-Arlington-Alexandria, DC-VA-MD-WV",240090011,0.040797,38.097046,24009,30500
246,48,Texas,203,Harrison,32220,"Marshall, TX",482030002,0.039419,37.517570,48203,30980


In [ ]:
# Let's also check rows where the EPA CBSA was unknown (we flagged these earlier, remember?)
epa_ozone[epa_ozone['cbsaname'] == 'UNKNOWN']
# When I hit the calculator icon, I see that there are values in cbsacode_y (from the Bureau) that we could fill in Unknown CBSAs for.

,fipsstatecode,state,fipscountycode,county,cbsacode_x,cbsaname,Site ID,Ozone Concentration (ppm),Daily AQI Value,CountyFIPS,cbsacode_y
8,42,Pennsylvania,117,Tioga,<NA>,UNKNOWN,421174000,0.040408,37.745834,42117,<NA>
34,42,Pennsylvania,59,Greene,<NA>,UNKNOWN,420590002,0.042918,40.936073,42059,<NA>
78,56,Wyoming,19,Johnson,<NA>,UNKNOWN,560190004,0.042542,39.593220,56019,<NA>
80,56,Wyoming,23,Lincoln,<NA>,UNKNOWN,560230004,0.045726,42.768570,56023,<NA>
83,56,Wyoming,3,Big Horn,<NA>,UNKNOWN,560030002,0.040147,37.229916,56003,<NA>
...,...,...,...,...,...,...,...,...,...,...,...
1209,29,Missouri,137,Monroe,<NA>,UNKNOWN,291370001,0.036112,33.430790,29137,<NA>
1210,29,Missouri,157,Perry,<NA>,UNKNOWN,291570001,0.044354,42.896843,29157,<NA>
1213,29,Missouri,186,Sainte Genevieve,<NA>,UNKNOWN,291860005,0.043559,41.668068,29186,<NA>
1220,29,Missouri,39,Cedar,<NA>,UNKNOWN,290390001,0.044898,43.217213,29039,<NA>


In [ ]:
# Okay : I'm going to make a CBSA conversion table. I'm going to prioritize official/updated values from the Bureau file, so that it'll be standardized across all EPA files, and so we can apply it to the CDC data in a more smooth way later.

# Starting with prioritizing the Bureau conversion codes
cbsa_lookup = conversion_dropped[['CountyFIPS', 'cbsacode']].copy()
cbsa_lookup = cbsa_lookup.rename(columns={'cbsacode': 'cbsa_final'})

# Okay - IN THE CASE where the Bureau is missing a CBSA code but the EPA has it, I'm goign to fallback onto EPA CBSAs from the two tables that do have it:
epa_cbsa_fallback = pd.concat([epa_no2[['CountyFIPS', 'cbsacode_x']], epa_ozone[['CountyFIPS', 'cbsacode_x']]]).drop_duplicates().dropna()

# Merge fallback in
cbsa_lookup = cbsa_lookup.merge(
    epa_cbsa_fallback, on='CountyFIPS', how='outer')

# I'll be prioritizing the Bureau values that are updated, but will fallback to EPA values if missing.
cbsa_lookup['cbsa_final'] = cbsa_lookup['cbsa_final'].fillna(cbsa_lookup['cbsacode_x'])

# Cleanup to ensure no duplicates.
cbsa_lookup = cbsa_lookup[['CountyFIPS', 'cbsa_final']].dropna().drop_duplicates()

# Alright, now that we have our cleaned lookup table let's merge the fixed CountyFIPS table to each of the tables, including county_summary (has our CDC data)
epa_pm25 = epa_pm25.merge(cbsa_lookup, on='CountyFIPS', how='left')
epa_no2 = epa_no2.merge(cbsa_lookup, on='CountyFIPS', how='left')
epa_ozone = epa_ozone.merge(cbsa_lookup, on='CountyFIPS', how='left')
county_summary = county_summary.merge(cbsa_lookup, on='CountyFIPS', how='left')

In [ ]:
county_summary
#Checking, I do see cbsa_final added. Nice!

,CountyFIPS,total_population,total_pop_18plus,chronic_obstructive_pulmonary_disease_among_adults_crude_prev,current_asthma_among_adults_crude_prev,stroke_among_adults_crude_prev,chronic_obstructive_pulmonary_disease_among_adults_adj_prev,current_asthma_among_adults_adj_prev,stroke_among_adults_adj_prev,state,county,cbsacode,cbsa_final
0,1001,58805,44523,8.382353,10.447059,3.911765,6.408058,7.948419,2.991839,Alabama,Autauga,33860,33860
1,1003,231767,182471,8.716279,10.025581,4.002326,6.959422,7.981697,3.199065,Alabama,Baldwin,19300,19300
2,1005,25223,20134,11.800000,11.455556,6.155556,9.410735,9.113765,4.905988,Alabama,Barbour,21640,21640
3,1007,22293,17533,10.825000,10.937500,4.825000,8.463181,8.556941,3.778982,Alabama,Bibb,13820,13820
4,1009,59134,45403,10.362500,10.525000,4.368750,7.954290,8.077116,3.354139,Alabama,Blount,13820,13820
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3138,56037,42272,31070,7.753846,10.676923,3.376923,5.815697,7.972194,2.533487,Wyoming,Sweetwater,40540,40540
3139,56039,23331,18755,5.185714,9.200000,2.557143,4.205425,7.450143,2.075455,Wyoming,Teton,27220,27220
3140,56041,20450,14627,7.550000,10.433333,3.266667,5.436582,7.500735,2.351915,Wyoming,Uinta,21740,21740
3141,56043,7685,5846,9.200000,10.566667,4.333333,7.002676,8.035279,3.301182,Wyoming,Washakie,<NA>,<NA>


In [ ]:
# I'm going to drop unnecessary columns - we can delete the individual EPA/Bureau columns since we have a unified column now.
epa_ozone = epa_ozone.drop(columns=['cbsacode_x', 'cbsacode_y'], errors='ignore')
epa_no2 = epa_no2.drop(columns=['cbsacode_x', 'cbsacode_y'], errors='ignore')

# I'm now going to drop cbsacode from CDC and epa_pm25 datasets
county_summary = county_summary.drop(columns=['cbsacode'], errors='ignore')
epa_pm25 = epa_pm25.drop(columns=['cbsacode'], errors='ignore')

# I'm doing this to rename cbsa_final to cbsacode in all datasets
epa_ozone = epa_ozone.rename(columns={'cbsa_final': 'cbsacode'})
epa_no2 = epa_no2.rename(columns={'cbsa_final': 'cbsacode'})
county_summary = county_summary.rename(columns={'cbsa_final': 'cbsacode'})
epa_pm25 = epa_pm25.rename(columns={'cbsa_final': 'cbsacode'})

# Now all datasets have the cbsacode column and unnecessary CBSA columns have been removed, let's check:
epa_ozone

,fipsstatecode,state,fipscountycode,county,cbsaname,Site ID,Ozone Concentration (ppm),Daily AQI Value,CountyFIPS,cbsacode
0,42,Pennsylvania,1,Adams,"Gettysburg, PA",420010001,0.042152,39.418034,42001,23900
1,42,Pennsylvania,1,Adams,"Gettysburg, PA",420019991,0.039752,37.225353,42001,23900
2,42,Pennsylvania,101,Philadelphia,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",421010004,0.035149,33.146553,42101,37980
3,42,Pennsylvania,101,Philadelphia,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",421010024,0.039518,38.563380,42101,37980
4,42,Pennsylvania,101,Philadelphia,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",421010048,0.037885,36.618910,42101,37980
...,...,...,...,...,...,...,...,...,...,...
1232,46,South Dakota,29,Codington,"Watertown, SD",460290002,0.039571,37.423077,46029,47980
1233,46,South Dakota,33,Custer,"Rapid City, SD",460330132,0.044179,41.940170,46033,39660
1234,46,South Dakota,71,Jackson,UNKNOWN,460710001,0.042738,40.543663,46071,<NA>
1235,46,South Dakota,93,Meade,"Rapid City, SD",460930001,0.041243,38.365920,46093,39660


In [ ]:
# Let's also check to ensure these cbsacodes are in our nhtslookup file, and see which ones do not match.
discrepant_cbsacodes = nhtslookup[~nhtslookup['cbsacode'].isin(cbsa_lookup['cbsa_final'])]

In [ ]:
# Okay since there's rural area codes custom made by NHTS that aren't integers, let's filter out any rows where cbsacode has any letters or non-numeric characters.
# Note that I plan to delete these later anyway, I just dont want them to intefere with our check now
discrepant_cbsacodes = discrepant_cbsacodes[discrepant_cbsacodes['cbsacode'].astype(str).str.isnumeric()]

# Now that we've removed non-numeric cbsacodes, we can safely convert to int64and check if the cbsacodes are in our final lookup and check which ones don't match
discrepant_cbsacodes['cbsacode'] = discrepant_cbsacodes['cbsacode'].astype(int)
discrepancies = discrepant_cbsacodes[~discrepant_cbsacodes['cbsacode'].isin(cbsa_lookup['cbsa_final'])]
print(discrepancies)

     index   zone_id                      zone_name states  cbsacode
55      56  15680_MD  California-Lexington Park, MD     MD     15680
84      85  17460_OH           Cleveland-Elyria, OH     OH     17460
118    119  20590_NC                Yanceyville, NC     NC     20590
249    250  31460_CA                     Madera, CA     CA     31460
295    296  36140_NJ                 Ocean City, NJ     NJ     36140
324    325  39140_AZ                   Prescott, AZ     AZ     39140
399    400  45540_FL               The Villages, FL     FL     45540
437    438  49060_AZ                    Nogales, AZ     AZ     49060


Okay so I checked each of these for why they'd be in NHTS but not in our final cbsa data. It looks like ocean city, yanceyville, and madera are really not appropriate to give cbsa codes to since they are more metropolitan areas. Interestingly, the LARGER region they belong to (ocean in New Jersey, Yancey, and Madera) DO have proper cbsacodes even in our cbsa_final. However, given that everything except Madera is explicitly referenced as the smaller counties here, I'm going to remove them just because I don't want to assume they are a CBSA. Madera could be referring properly to the county, so I'm going to just adjust the CBSA code provided in the NHTS file to the one in our cbsa_final column (when I searched for the CBSA code for Madera in previous tables, it turned out to be 35620) so I'll replace it with that.


In [ ]:
# Okay I'm going to replace Madera, remove discrepancies, and then also remove any of the custom cbsacodes NHTS provided (which is marked by a string)
nhtslookup.loc[nhtslookup['zone_name'] == 'Madera, CA', 'cbsacode'] = 35620
nhtslookup = nhtslookup[~nhtslookup['cbsacode'].isin(discrepant_cbsacodes['cbsacode'])]
nhtslookup = nhtslookup[nhtslookup['cbsacode'].str.fullmatch(r'\d+', na=False)]

## Transforming NHTS Transportation Data
The goals here are to ensure all my CBSA codes align, collapsing all subgrouped columns of distances into a either a short or long column (so adding distance 0-50 into one column and then 50-300+ into another) and then removing the individual columns to save space. Also, NHTS data is provided in "zones" we'll have to convert to CBSAs.

I want to aggregate the NHTS data such that when I query for a given CBSA code, I'll be able to see information on outbound trips, inbound trips, etc.

This will require me to aggregate by zone ID individually for both origin and destination, and keeping the sums separate and then adding them as columns to my final NHTS dataframe.

In [ ]:
# But before I do ANY OF THAT: I'm not just going to rely on their flagging system from earlier (which marks suspicious rows as 1), I'm going to also manually remove rows where annual_total_trips is zero. But first I want to see how many rows that holds true for to see how many would actually be dropped if I did that to make sure I'm not losing too much data.
passenger_cleaned[passenger_cleaned['annual_total_trips'] == 0]

,origin_zone_id,destination_zone_id,origin_zone_name,destination_zone_name,origin_state,destination_state,annual_total_trips,mode_air,mode_rail,mode_vehicle,...,purpose_nonwork,distance_0_10,distance_10_25,distance_25_50,distance_50_75,distance_75_100,distance_100_150,distance_150_300,distance_gt300,flag
12,10180_TX,11260_AK,"Abilene, TX","Anchorage, AK",TX,AK,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
95,10180_TX,18700_OR,"Abilene, TX","Corvallis, OR",TX,OR,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
118,10180_TX,20590_NC,"Abilene, TX","Yanceyville, NC",TX,NC,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
119,10180_TX,20700_PA,"Abilene, TX","East Stroudsburg, PA",TX,PA,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
131,10180_TX,21820_AK,"Abilene, TX","Fairbanks, AK",TX,AK,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
339733,RWY3_WY,48260_WV,WY-NonMSA areas (SW),"Weirton-Steubenville, WV-OH",WY,WV,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
339742,RWY3_WY,49020_WV,WY-NonMSA areas (SW),"Winchester, VA-WV",WY,WV,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
339753,RWY3_WY,RAK1_AK,WY-NonMSA areas (SW),AK-NonMSA areas,WY,AK,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
339770,RWY3_WY,RFL2_FL,WY-NonMSA areas (SW),"Union County, FL",WY,FL,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Ok so the command above prints out 24,456 rows. Out of 294841 rows, dropping these would mean we would retain 270,385 rows. That's still 91% of the initial non-flagged entries. I think overall the sample size of 270385 is still sufficiently huge. Also, many of the printed entries seem to be in rural areas (which I will probably drop later since many don't even have CBSA codes anyway).Given all of this, I'm going to drop these rows, I think it's justified since I'm still retaining over 90% of my initial data.

In [ ]:
passenger_cleaned = passenger_cleaned[passenger_cleaned['annual_total_trips'] > 0].reset_index(drop=True)
passenger_cleaned = passenger_cleaned.drop(columns='flag')
print(passenger_cleaned.shape[0])
print(passenger_cleaned.dtypes)
# Okay yup as expected, 270385 rows left.

270385
origin_zone_id           object
destination_zone_id      object
origin_zone_name         object
destination_zone_name    object
origin_state             object
destination_state        object
annual_total_trips        int64
mode_air                  int64
mode_rail                 int64
mode_vehicle              int64
mode_atf                  int64
purpose_work              int64
purpose_nonwork           int64
distance_0_10             int64
distance_10_25            int64
distance_25_50            int64
distance_50_75            int64
distance_75_100           int64
distance_100_150          int64
distance_150_300          int64
distance_gt300            int64
dtype: object


In [ ]:
# Alright now I want to sum up all distance columns into either low or high. I'm going to classify low as 0-50 miles, and anything above that as high.
# Add distance_low column: sum of short-distance trips (in percentage)
passenger_cleaned['distance_low'] = (
    passenger_cleaned['distance_0_10'] +
    passenger_cleaned['distance_10_25'] +
    passenger_cleaned['distance_25_50'])

# Add distance_high column: sum of longer-distance trips (in percentage)
passenger_cleaned['distance_high'] = (
    passenger_cleaned['distance_50_75'] +
    passenger_cleaned['distance_75_100'] +
    passenger_cleaned['distance_100_150'] +
    passenger_cleaned['distance_150_300'] +
    passenger_cleaned['distance_gt300'])

# I'm then going to drop those distance columns and only keep the low/high columns.
to_drop_cols = ['distance_0_10', 'distance_10_25', 'distance_25_50', 'distance_50_75', 'distance_75_100', 'distance_100_150', 'distance_150_300', 'distance_gt300']

passenger_cleaned = passenger_cleaned.drop(columns=to_drop_cols) # actually dropping
# Let's check
passenger_cleaned

,origin_zone_id,destination_zone_id,origin_zone_name,destination_zone_name,origin_state,destination_state,annual_total_trips,mode_air,mode_rail,mode_vehicle,mode_atf,purpose_work,purpose_nonwork,distance_low,distance_high
0,10180_TX,10180_TX,"Abilene, TX","Abilene, TX",TX,TX,179452461,0,0,163715930,15736531,54695451,124757010,178781629,670832
1,10180_TX,10420_OH,"Abilene, TX","Akron, OH",TX,OH,71,0,0,71,0,31,40,0,71
2,10180_TX,10500_GA,"Abilene, TX","Albany, GA",TX,GA,105,0,0,105,0,86,19,0,105
3,10180_TX,10580_NY,"Abilene, TX","Albany-Schenectady-Troy, NY",TX,NY,37,0,0,37,0,11,26,0,37
4,10180_TX,10740_NM,"Abilene, TX","Albuquerque, NM",TX,NM,4022,323,0,3699,0,2429,1593,0,4022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
270380,RWY3_WY,RWI3_WI,WY-NonMSA areas (SW),WI-NonMSA areas (N),WY,WI,324,0,0,324,0,86,238,0,324
270381,RWY3_WY,RWV2_WV,WY-NonMSA areas (SW),WV-NonMSA areas (NW),WY,WV,72,0,0,72,0,54,18,0,72
270382,RWY3_WY,RWY1_WY,WY-NonMSA areas (SW),WY-NonMSA areas (E),WY,WY,308418,0,0,306236,2182,141551,166867,66201,242217
270383,RWY3_WY,RWY2_WY,WY-NonMSA areas (SW),WY-NonMSA areas (NW),WY,WY,462361,1180,0,458665,2516,190788,271573,195191,267170


In [ ]:
# Alright now I'm going to map origin_zone_id to cbsa code based off of our lookup file and create a new column called origincbsa
passenger_cleaned['origincbsa'] = passenger_cleaned['origin_zone_id'].map(
    nhtslookup.set_index('zone_id')['cbsacode'])

# And I'm going to do the same for our destinationcbsa.
passenger_cleaned['destinationcbsa'] = passenger_cleaned['destination_zone_id'].map(
    nhtslookup.set_index('zone_id')['cbsacode'])

# Prior to doing this I want to check how many rows we'll lose at this point because they are rural (all rural codes start with letters):
sum(passenger_cleaned['origin_zone_id'].str[0].str.isalpha() | passenger_cleaned['destination_zone_id'].str[0].str.isalpha())

115312

In [ ]:
# Note: 115312 rural rows being removed may seem like a lot, but that still leaves us with 150,000+ rows. I didn't want to include rural areas because there's no data on them in EPA/CDC or way to merge them. So I'm going to remove them, even though it's a lot.
# If there's an NA either in origincbsa or destinationcbsa, indicating that it's not present in our lookup table (since we removed all the rural codes & mismatched codes), I want to remove that row.
passenger_cleaned = passenger_cleaned.dropna(subset=['origincbsa', 'destinationcbsa'])
passenger_cleaned.dtypes

,0
origin_zone_id,object
destination_zone_id,object
origin_zone_name,object
destination_zone_name,object
origin_state,object
destination_state,object
annual_total_trips,int64
mode_air,int64
mode_rail,int64
mode_vehicle,int64


In [ ]:
# I'm going to convert the cbsas into integers to keep formatting consistent
passenger_cleaned['origincbsa'] = passenger_cleaned['origincbsa'].astype('int64')
passenger_cleaned['destinationcbsa'] = passenger_cleaned['destinationcbsa'].astype('int64')

# And remove any rows where the cbsa was in our discrepancies (so rural counties, mismatches, etc.).
passenger_cleaned = passenger_cleaned[
    ~passenger_cleaned['origincbsa'].isin(discrepancies['cbsacode']) &
    ~passenger_cleaned['destinationcbsa'].isin(discrepancies['cbsacode'])]

# When this loads I manually checked (using the calculator icon) - no rural/letter cbsacodes are present, and neither are the ones that did not match initially! Madera is also in there, but with the proper cbsa code this time.
passenger_cleaned

<ipython-input-57-c5860f31aab5>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  passenger_cleaned['origincbsa'] = passenger_cleaned['origincbsa'].astype('int64')
<ipython-input-57-c5860f31aab5>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  passenger_cleaned['destinationcbsa'] = passenger_cleaned['destinationcbsa'].astype('int64')


,origin_zone_id,destination_zone_id,origin_zone_name,destination_zone_name,origin_state,destination_state,annual_total_trips,mode_air,mode_rail,mode_vehicle,mode_atf,purpose_work,purpose_nonwork,distance_low,distance_high,origincbsa,destinationcbsa
0,10180_TX,10180_TX,"Abilene, TX","Abilene, TX",TX,TX,179452461,0,0,163715930,15736531,54695451,124757010,178781629,670832,10180,10180
1,10180_TX,10420_OH,"Abilene, TX","Akron, OH",TX,OH,71,0,0,71,0,31,40,0,71,10180,10420
2,10180_TX,10500_GA,"Abilene, TX","Albany, GA",TX,GA,105,0,0,105,0,86,19,0,105,10180,10500
3,10180_TX,10580_NY,"Abilene, TX","Albany-Schenectady-Troy, NY",TX,NY,37,0,0,37,0,11,26,0,37,10180,10580
4,10180_TX,10740_NM,"Abilene, TX","Albuquerque, NM",TX,NM,4022,323,0,3699,0,2429,1593,0,4022,10180,10740
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204832,49740_AZ,49620_PA,"Yuma, AZ","York-Hanover, PA",AZ,PA,141,0,0,141,0,39,102,0,141,49740,49620
204833,49740_AZ,49660_OH,"Yuma, AZ","Youngstown-Warren-Boardman, OH-PA",AZ,OH,331,0,0,331,0,230,101,0,331,49740,49660
204834,49740_AZ,49660_PA,"Yuma, AZ","Youngstown-Warren-Boardman, OH-PA",AZ,PA,122,0,0,122,0,88,34,0,122,49740,49660
204835,49740_AZ,49700_CA,"Yuma, AZ","Yuba City, CA",AZ,CA,489,0,0,489,0,215,274,0,489,49740,49700


Alright! Time to sum up our NHTS data. Again, I wanted it to be formatted such that each CBSA would be its own row, with outbound aggregated values and inbound aggregated values. This is why I kept the raw number of trips rather than the percentage, so I could recalculate! See below.

In [ ]:
# Step 1: Aggregate outbound data (based on destinationcbsa) and calculate percentage
outbound_agg = passenger_cleaned.groupby('destinationcbsa').agg(
    outbound_annual_trips=('annual_total_trips', 'sum'),
    outbound_mode_air=('mode_air', 'sum'),
    outbound_mode_rail=('mode_rail', 'sum'),
    outbound_mode_vehicle=('mode_vehicle', 'sum'),
    outbound_mode_atf=('mode_atf', 'sum'),
    outbound_purpose_work=('purpose_work', 'sum'),
    outbound_purpose_nonwork=('purpose_nonwork', 'sum'),
    outbound_distance_low=('distance_low', 'sum'),
    outbound_distance_high=('distance_high', 'sum')).reset_index()

# Calculate percentages by dividing sum by outbound_annual_trips
outbound_agg['outbound_mode_air_pct'] = outbound_agg['outbound_mode_air'] / outbound_agg['outbound_annual_trips']
outbound_agg['outbound_mode_rail_pct'] = outbound_agg['outbound_mode_rail'] / outbound_agg['outbound_annual_trips']
outbound_agg['outbound_mode_vehicle_pct'] = outbound_agg['outbound_mode_vehicle'] / outbound_agg['outbound_annual_trips']
outbound_agg['outbound_mode_atf_pct'] = outbound_agg['outbound_mode_atf'] / outbound_agg['outbound_annual_trips']
outbound_agg['outbound_purpose_work_pct'] = outbound_agg['outbound_purpose_work'] / outbound_agg['outbound_annual_trips']
outbound_agg['outbound_purpose_nonwork_pct'] = outbound_agg['outbound_purpose_nonwork'] / outbound_agg['outbound_annual_trips']
outbound_agg['outbound_distance_low_pct'] = outbound_agg['outbound_distance_low'] / outbound_agg['outbound_annual_trips']
outbound_agg['outbound_distance_high_pct'] = outbound_agg['outbound_distance_high'] / outbound_agg['outbound_annual_trips']

# Drop raw sum columns after calculating percentages
outbound_agg.drop(columns=[
    'outbound_mode_air', 'outbound_mode_rail', 'outbound_mode_vehicle',
    'outbound_mode_atf', 'outbound_purpose_work', 'outbound_purpose_nonwork',
    'outbound_distance_low', 'outbound_distance_high'
], inplace=True)

# Step 2: Aggregate inbound data (based on origincbsa) and calculate percentage
inbound_agg = passenger_cleaned.groupby('origincbsa').agg(
    inbound_annual_trips=('annual_total_trips', 'sum'),
    inbound_mode_air=('mode_air', 'sum'),
    inbound_mode_rail=('mode_rail', 'sum'),
    inbound_mode_vehicle=('mode_vehicle', 'sum'),
    inbound_mode_atf=('mode_atf', 'sum'),
    inbound_purpose_work=('purpose_work', 'sum'),
    inbound_purpose_nonwork=('purpose_nonwork', 'sum'),
    inbound_distance_low=('distance_low', 'sum'),
    inbound_distance_high=('distance_high', 'sum')).reset_index()

# Calculate percentages by dividing sum by inbound_annual_trips
inbound_agg['inbound_mode_air_pct'] = inbound_agg['inbound_mode_air'] / inbound_agg['inbound_annual_trips']
inbound_agg['inbound_mode_rail_pct'] = inbound_agg['inbound_mode_rail'] / inbound_agg['inbound_annual_trips']
inbound_agg['inbound_mode_vehicle_pct'] = inbound_agg['inbound_mode_vehicle'] / inbound_agg['inbound_annual_trips']
inbound_agg['inbound_mode_atf_pct'] = inbound_agg['inbound_mode_atf'] / inbound_agg['inbound_annual_trips']
inbound_agg['inbound_purpose_work_pct'] = inbound_agg['inbound_purpose_work'] / inbound_agg['inbound_annual_trips']
inbound_agg['inbound_purpose_nonwork_pct'] = inbound_agg['inbound_purpose_nonwork'] / inbound_agg['inbound_annual_trips']
inbound_agg['inbound_distance_low_pct'] = inbound_agg['inbound_distance_low'] / inbound_agg['inbound_annual_trips']
inbound_agg['inbound_distance_high_pct'] = inbound_agg['inbound_distance_high'] / inbound_agg['inbound_annual_trips']

# Drop raw sum columns after calculating percentages
inbound_agg.drop(columns=[
    'inbound_mode_air', 'inbound_mode_rail', 'inbound_mode_vehicle',
    'inbound_mode_atf', 'inbound_purpose_work', 'inbound_purpose_nonwork',
    'inbound_distance_low', 'inbound_distance_high'
], inplace=True)

# Merge outbound and inbound tables based on cbsa_code
merged_nhts_data = outbound_agg.merge(inbound_agg, left_on='destinationcbsa', right_on='origincbsa', how='outer')

# I just used origincbsa, but I'm going to drop it since it's now redundant
merged_nhts_data.drop(columns=['origincbsa'], inplace=True)

# I'm going to rename columns
merged_nhts_data.rename(columns={
    'destinationcbsa': 'cbsacode'}, inplace=True)

# Final table with outbound and inbound data and percentages - let's see if all of this worked
print(merged_nhts_data.head())

   cbsacode  outbound_annual_trips  outbound_mode_air_pct  \
0     10180              182042332               0.000826   
1     10420              702867310               0.000555   
2     10500              163588267               0.000518   
3     10540              121896038               0.000000   
4     10580              870348816               0.001720   

   outbound_mode_rail_pct  outbound_mode_vehicle_pct  outbound_mode_atf_pct  \
0                0.000000                   0.912730               0.086444   
1                0.000001                   0.877631               0.121812   
2                0.000000                   0.900179               0.099303   
3                0.000242                   0.840219               0.159539   
4                0.000632                   0.779819               0.217829   

   outbound_purpose_work_pct  outbound_purpose_nonwork_pct  \
0                   0.306203                      0.693797   
1                   0.280283      

That was definitely the trickiest thing so far, but looks like the data is formatted as I intended (each cbsacode as its own row, aggregated outbound and inbound info as columns). Now you can see how trip distance, mode, and number varied if it was outbound/inbound for each column.

## Transforming EPA Data
Again, just like we did with everything before, I'm going to aggregate everything to CountyFIPS (rather than site). I'm going to use the average ozone concentration across sites for a given CountyFIPs and the average AQI (air quality index). # Note I added in later: I decided to include AQI just as another metric. Note that since sites are different, and AQI seems to vary, I don't think it's fair to just merge AQIs all into one. Different sensors (ozone-specific, no2-specific, etc.) seem to collect various AQI metrics, so depending on what the end-user is looking for, they might choose one AQI metric over the other. Thought it'd be nice to include all in my final RDB.

In [ ]:
# Ozone: Aggregate to get mean ozone concentration and AQI by CountyFIPS
epa_ozone_agg = epa_ozone.groupby('CountyFIPS').agg(
    ozone_concentration=('Ozone Concentration (ppm)', 'mean'),
    ozone_AQI=('Daily AQI Value', 'mean')
).reset_index()

# NO2: Aggregate to get mean NO2 concentration and AQI by CountyFIPS
epa_no2_agg = epa_no2.groupby('CountyFIPS').agg(
    no2_concentration=('NO2 Concentration (ppb)', 'mean'),
    no2_AQI=('Daily AQI Value', 'mean')
).reset_index()

# PM2.5: Aggregate to get mean PM2.5 concentration and AQI by CountyFIPS
epa_pm25_agg = epa_pm25.groupby('CountyFIPS').agg(
    pm25_concentration=('PM2.5 Concentration (ug/m3)', 'mean'),
    pm25_AQI=('Daily AQI Value', 'mean')
).reset_index()

In [ ]:
# Okay time to make our big EPA table! I'm going to merge the three tables (Ozone, NO2, PM2.5) based on CountyFIPS
epamerg = epa_ozone_agg.merge(epa_no2_agg, on='CountyFIPS', how='outer') # joins ozone to no2
merged_epa = epamerg.merge(epa_pm25_agg, on='CountyFIPS', how='outer') # joins pm25 to the ozone-no2 table we just made

# Print the merged EPA data table
print(merged_epa.head())

# All right. nice. there's our EPA table, one row for each unique CountyFIPS. As expected, some CountyFIPS didn't have sites that collected certain pollutants.

   CountyFIPS  ozone_concentration  ozone_AQI  no2_concentration  no2_AQI  \
0        1003             0.039370  37.049380                NaN      NaN   
1        1027                  NaN        NaN                NaN      NaN   
2        1049             0.038920  36.366390                NaN      NaN   
3        1051             0.036412  33.936974                NaN      NaN   
4        1055             0.040882  38.457985                NaN      NaN   

   pm25_concentration   pm25_AQI  
0            7.317391  38.269566  
1            6.880869  35.921738  
2            6.855932  36.135593  
3                 NaN        NaN  
4            8.212079  41.415730  


In [ ]:
# Okay let's add CBSA codes now!
merged_epa = merged_epa.merge(cbsa_lookup, on='CountyFIPS', how='left')
merged_epa.rename(columns={'cbsa_final': 'cbsacode'}, inplace=True)

## Creating a county_info table

In [ ]:
#Ok now I'm going to take all state/county info from individual EPA tables, and then just add to epa_merged. I want to be able to have all of this because I'm shortly going to create a county_info table,
# And to make a county_info table I'll need state & county info for each provided CountyFIPS across the board.
epa_meta = pd.concat([
    epa_ozone[['CountyFIPS', 'state', 'county']],
    epa_no2[['CountyFIPS', 'state', 'county']],
    epa_pm25[['CountyFIPS', 'state', 'county']]
]).drop_duplicates(subset='CountyFIPS')

merged_epa = merged_epa.merge(epa_meta, on='CountyFIPS', how='left')
merged_epa

,CountyFIPS,ozone_concentration,ozone_AQI,no2_concentration,no2_AQI,pm25_concentration,pm25_AQI,cbsacode,state,county
0,1003,0.039370,37.049380,NaN,NaN,7.317391,38.269566,19300,Alabama,Baldwin
1,1027,NaN,NaN,NaN,NaN,6.880869,35.921738,<NA>,Alabama,Clay
2,1049,0.038920,36.366390,NaN,NaN,6.855932,36.135593,22840,Alabama,DeKalb
3,1051,0.036412,33.936974,NaN,NaN,NaN,NaN,33860,Alabama,Elmore
4,1055,0.040882,38.457985,NaN,NaN,8.212079,41.415730,23460,Alabama,Etowah
...,...,...,...,...,...,...,...,...,...,...
979,72061,NaN,NaN,NaN,NaN,6.888288,34.918920,41980,Puerto Rico,Guaynabo
980,72097,0.019081,17.622581,NaN,NaN,8.271382,39.527332,32420,Puerto Rico,Mayagnez
981,72113,NaN,NaN,NaN,NaN,6.661538,33.461540,38660,Puerto Rico,Ponce
982,78010,NaN,NaN,NaN,NaN,6.726923,34.423077,<NA>,Virgin Islands,St Croix


Okay now I can finally create a sort of county_info table that will have information linking CountyFIPS to state, county, and cbsacode ACROSS all of my tables. I know someone might ask why I didn't do this initially: I waited to do this until now, as opposed to doing it earlier, because I wanted to transform, aggregate, and standardize CountyFIPS across all data to see what was even left prior to creating a sort of combined lookup/county reference table. I'll be pushing this final version into my RDB!

In [ ]:
# First, I'm going to gather all uniqeu CountyFIPS values from CDC and EPA data
all_countyfips = pd.Series(
    pd.concat([county_summary['CountyFIPS'], merged_epa['CountyFIPS']]).unique(),
    name='CountyFIPS')

# Then I'm going to merge to get respective county/state/cbsacode info from county_summary (CDC data)
# ^ Note I just chose CDC data initially because CDC data is the most comprehensive in terms of number of counties covered.
county_info = all_countyfips.to_frame().merge(
    county_summary[['CountyFIPS', 'state', 'county', 'cbsacode']],
    on='CountyFIPS',
    how='left')

# Of course following that up with info using merged_epa as backup
county_info = county_info.merge(
    merged_epa[['CountyFIPS', 'state', 'county', 'cbsacode']],
    on='CountyFIPS',
    how='left',
    suffixes=('_cdc', '_epa')) # did this so the columns wouldn't duplicate since they're called the same thing

county_info

,CountyFIPS,state_cdc,county_cdc,cbsacode_cdc,state_epa,county_epa,cbsacode_epa
0,1001,Alabama,Autauga,33860,NaN,NaN,<NA>
1,1003,Alabama,Baldwin,19300,Alabama,Baldwin,19300
2,1005,Alabama,Barbour,21640,NaN,NaN,<NA>
3,1007,Alabama,Bibb,13820,NaN,NaN,<NA>
4,1009,Alabama,Blount,13820,NaN,NaN,<NA>
...,...,...,...,...,...,...,...
3156,72061,NaN,NaN,<NA>,Puerto Rico,Guaynabo,41980
3157,72097,NaN,NaN,<NA>,Puerto Rico,Mayagnez,32420
3158,72113,NaN,NaN,<NA>,Puerto Rico,Ponce,38660
3159,78010,NaN,NaN,<NA>,Virgin Islands,St Croix,<NA>


In [ ]:
# Then I'm going to fill in missing values from EPA if CDC data didn't have it
county_info['state'] = county_info['state_cdc'].combine_first(county_info['state_epa'])
county_info['county'] = county_info['county_cdc'].combine_first(county_info['county_epa'])
county_info['cbsacode'] = county_info['cbsacode_cdc'].combine_first(county_info['cbsacode_epa'])
county_info

,CountyFIPS,state_cdc,county_cdc,cbsacode_cdc,state_epa,county_epa,cbsacode_epa,state,county,cbsacode
0,1001,Alabama,Autauga,33860,NaN,NaN,<NA>,Alabama,Autauga,33860
1,1003,Alabama,Baldwin,19300,Alabama,Baldwin,19300,Alabama,Baldwin,19300
2,1005,Alabama,Barbour,21640,NaN,NaN,<NA>,Alabama,Barbour,21640
3,1007,Alabama,Bibb,13820,NaN,NaN,<NA>,Alabama,Bibb,13820
4,1009,Alabama,Blount,13820,NaN,NaN,<NA>,Alabama,Blount,13820
...,...,...,...,...,...,...,...,...,...,...
3156,72061,NaN,NaN,<NA>,Puerto Rico,Guaynabo,41980,Puerto Rico,Guaynabo,41980
3157,72097,NaN,NaN,<NA>,Puerto Rico,Mayagnez,32420,Puerto Rico,Mayagnez,32420
3158,72113,NaN,NaN,<NA>,Puerto Rico,Ponce,38660,Puerto Rico,Ponce,38660
3159,78010,NaN,NaN,<NA>,Virgin Islands,St Croix,<NA>,Virgin Islands,St Croix,<NA>


In [ ]:
# Notice how in the frame that printed above there is state/county/cbsacode data for both CDC and EPA. Since we've gotten our combined column, Then I'm going to drop these columns now that the finalized versions are all good
county_info = county_info.drop(columns=['state_cdc', 'state_epa', 'county_cdc', 'county_epa', 'cbsacode_cdc', 'cbsacode_epa'])

# Let's see if that worked!
print(county_info.head())
# It did! Each CountyFIPS has a unique state, county, and cbsacode.

   CountyFIPS    state   county  cbsacode
0        1001  Alabama  Autauga     33860
1        1003  Alabama  Baldwin     19300
2        1005  Alabama  Barbour     21640
3        1007  Alabama     Bibb     13820
4        1009  Alabama   Blount     13820


# Loading Data

In [ ]:
# Helper functions
import psycopg2

# Make our connection/cursor function
AWS_host_name = "[REDACTED]finalprojectrdb.cjmqwq4m077e.us-east-2.rds.amazonaws.com"
AWS_dbname = "postgres"
AWS_user_name = "postgres"
AWS_password = "[imnotgivingyoumypasswordsorry]$"

def get_conn_cur(): # define function name and arguments (there aren't any)
  # Make a connection
  conn = psycopg2.connect(
    host=AWS_host_name,
    database=AWS_dbname,
    user=AWS_user_name,
    password=AWS_password,
    port='5432')

  cur = conn.cursor()   # Make a cursor after

  return(conn, cur)   # Return both the connection and the cursor

def get_table_names():
  conn, cur = get_conn_cur() # get connection and cursor

  # query to get table names
  table_name_query = """SELECT table_name FROM information_schema.tables
       WHERE table_schema = 'public' """

  cur.execute(table_name_query) # execute
  my_data = cur.fetchall() # fetch results

  cur.close() #close cursor
  conn.close() # close connection

  return(my_data) # return your fetched results


def get_column_names(table_name): # arguement of table_name
  conn, cur = get_conn_cur() # get connection and cursor

  # Now select column names while inserting the table name into the WERE
  column_name_query =  """SELECT column_name FROM information_schema.columns
       WHERE table_name = '%s' """ %table_name

  cur.execute(column_name_query) # exectue
  my_data = cur.fetchall() # store

  cur.close() # close
  conn.close() # close

  return(my_data) # return

def run_query(query_string):

  conn, cur = get_conn_cur() # get connection and cursor

  cur.execute(query_string) # executing string as before

  my_data = cur.fetchall() # fetch query data as before

  # here we're extracting the 0th element for each item in cur.description
  colnames = [desc[0] for desc in cur.description]

  cur.close() # close
  conn.close() # close

  return(colnames, my_data) # return column names AND data

# make sql_head function

def sql_head(table_name):
  conn, cur = get_conn_cur() # get connection and cursor

  table_head_query = """SELECT * FROM %s
                          LIMIT 5""" %table_name

  cur.execute(table_head_query) # exectue
  my_data = cur.fetchall() # store
  colnames = [desc[0] for desc in cur.description]

  cur.close() # close
  conn.close() # close

  df = pd.DataFrame(data = my_data, columns = colnames) # make into df

  return(df) # return

def my_drop_table(tab_name):
  conn, cur = get_conn_cur()
  tq = """DROP TABLE IF EXISTS %s CASCADE;""" %tab_name
  cur.execute(tq)
  conn.commit()

# Schema
The overall schema is designed to facilitate multi-table queries primarily by either CounyFIPS or cbsacode. The county_info table is my main anchor/reference here: CountyFIPS is its primary key. I also chose to use CountyFIPS as my primary key as opposed to cbsacode to preserve granularity/more detail (since there's more counties than CBSAs). This key is a foreign key in cdc_outcomes and epa_pollution tables to link health/polution data. Since multiple counties CAN belong to the same CBSA, it's not a primary or foreign key. I did this intentionally so you could query either a single county (via CountyFIPS) or an entire cbsacode and pull related data. See image in slides (had difficulty putting it in the notebook).

## Loading county_info
Again, this is going to serve as my anchor table so that when we query by, for example, countyFIPS, the respective state, county and cbsacode will show up along with other data. By centralizing this in one table I don't have to re-include/redundantly include state/county/cbsacode/countyFIPS in my other tables.

In [ ]:
county_info = county_info.fillna(np.nan).replace([np.nan], [None]) # Replacing with these values so that it's SQL format friendly
county_info # let's check to see the table looks good before we load it.

,CountyFIPS,state,county,cbsacode
0,1001,Alabama,Autauga,33860
1,1003,Alabama,Baldwin,19300
2,1005,Alabama,Barbour,21640
3,1007,Alabama,Bibb,13820
4,1009,Alabama,Blount,13820
...,...,...,...,...
3156,72061,Puerto Rico,Guaynabo,41980
3157,72097,Puerto Rico,Mayagnez,32420
3158,72113,Puerto Rico,Ponce,38660
3159,78010,Virgin Islands,St Croix,None


In [ ]:
my_drop_table('county_info') # drop the table if already exists

In [ ]:
# Connect and create table
conn, cur = get_conn_cur()
tq = """
CREATE TABLE county_info (
    CountyFIPS INTEGER NOT NULL PRIMARY KEY,
    state VARCHAR(255),
    county VARCHAR(255),
    cbsacode INTEGER
);
"""
cur.execute(tq)
conn.commit()
get_table_names()
county_tups = [tuple(x) for x in county_info.to_numpy()] # this is a shortcut that just tuples our columns and ensures they are in the right format
iq = """INSERT INTO county_info (""" + ','.join(county_info.columns) + """) VALUES (%s, %s, %s, %s);"""
cur.executemany(iq, county_tups)
conn.commit()

# Preview the inserted data
sql_head('county_info')
#nice, it worked.

,countyfips,state,county,cbsacode
0,1001,Alabama,Autauga,33860
1,1003,Alabama,Baldwin,19300
2,1005,Alabama,Barbour,21640
3,1007,Alabama,Bibb,13820
4,1009,Alabama,Blount,13820


## Loading CDC_OUTCOMES

This contains all my health outcomes from my CDC data. CountyFIPS serves as the primary key here!

In [ ]:
#Loading/quickly filtering county_summary & checking dtypes
county_summary = county_summary.fillna(np.nan).replace([np.nan], [None])
county_summary = county_summary.drop(columns=['state', 'county', 'cbsacode']) # don't need to include these anymore since we have county_info!
county_summary = county_summary.rename(columns={
    'chronic_obstructive_pulmonary_disease_among_adults_crude_prev': 'copd_crude_prev',
    'current_asthma_among_adults_crude_prev': 'asthma_crude_prev',
    'stroke_among_adults_crude_prev': 'stroke_crude_prev',
    'chronic_obstructive_pulmonary_disease_among_adults_adj_prev': 'copd_adj_prev',
    'current_asthma_among_adults_adj_prev': 'asthma_adj_prev',
    'stroke_among_adults_adj_prev': 'stroke_adj_prev'}) #decided to rename these because they were too long

county_summary.dtypes

,0
CountyFIPS,int64
total_population,int64
total_pop_18plus,int64
copd_crude_prev,float64
asthma_crude_prev,float64
stroke_crude_prev,float64
copd_adj_prev,float64
asthma_adj_prev,float64
stroke_adj_prev,float64


In [ ]:
my_drop_table('cdc_outcomes') # drop if it already exists

In [ ]:
conn, cur = get_conn_cur()
tq = """
CREATE TABLE cdc_outcomes (
    CountyFIPS INTEGER NOT NULL PRIMARY KEY,
    total_population INTEGER,
    total_pop_18plus INTEGER,
    copd_crude_prev FLOAT,
    asthma_crude_prev FLOAT,
    stroke_crude_prev FLOAT,
    copd_adj_prev FLOAT,
    asthma_adj_prev FLOAT,
    stroke_adj_prev FLOAT,
    FOREIGN KEY (CountyFIPS) REFERENCES county_info(CountyFIPS)
);
"""
cur.execute(tq)
conn.commit()
cdc_tups = [tuple(x.item() if hasattr(x, 'item') else x for x in row) for row in county_summary.to_numpy()] # handy simplified version of code that converts each row in county_summary into a tuple, and ensures scalar values are converted to proper python equivalents
placeholders = ','.join(['%s'] * len(county_summary.columns)) # since we dropped unnecessary columns, we can just use the length.
iq = f"""INSERT INTO cdc_outcomes ({','.join(county_summary.columns)}) VALUES ({placeholders});"""
cur.executemany(iq, cdc_tups)
conn.commit()

sql_head('cdc_outcomes')

,countyfips,total_population,total_pop_18plus,copd_crude_prev,asthma_crude_prev,stroke_crude_prev,copd_adj_prev,asthma_adj_prev,stroke_adj_prev
0,1001,58805,44523,8.382353,10.447059,3.911765,6.408058,7.948419,2.991839
1,1003,231767,182471,8.716279,10.025581,4.002326,6.959422,7.981697,3.199065
2,1005,25223,20134,11.800000,11.455556,6.155556,9.410735,9.113765,4.905988
3,1007,22293,17533,10.825000,10.937500,4.825000,8.463181,8.556941,3.778982
4,1009,59134,45403,10.362500,10.525000,4.368750,7.954290,8.077116,3.354139


## Loading in epa_pollution
This table has metrics on aggregated annual EPA pollution levels by countyFIPS, and the air quality index levels collected by sensor type in case an end user wants to look at specific AQIs and how they associate to the RESPECTIVE sensor (again, since there was variation in AQIs quite dramatically by sensor type in certain rows, I figured it'd be best not to just average them. Maybe they are different for a reason).  

In [ ]:
merged_epa = merged_epa.drop(columns=['state', 'county', 'cbsacode']) # again dropping these since we have county_info
merged_epa = merged_epa.fillna(np.nan).replace([np.nan], [None]) # making it SQL friendly
merged_epa.dtypes

,0
CountyFIPS,int64
ozone_concentration,object
ozone_AQI,object
no2_concentration,object
no2_AQI,object
pm25_concentration,object
pm25_AQI,object


In [ ]:
my_drop_table('epa_pollution') # dropping table if it already exists
conn, cur = get_conn_cur()
tq = """
CREATE TABLE epa_pollution (
    CountyFIPS INTEGER NOT NULL PRIMARY KEY,
    ozone_concentration FLOAT,
    ozone_AQI FLOAT,
    no2_concentration FLOAT,
    no2_AQI FLOAT,
    pm25_concentration FLOAT,
    pm25_AQI FLOAT,
    FOREIGN KEY (CountyFIPS) REFERENCES county_info(CountyFIPS)
);
"""
cur.execute(tq)
conn.commit()
epa_tups = [tuple(x.item() if hasattr(x, 'item') else x for x in row) for row in merged_epa.to_numpy()] # converting rows to tuples, making it python friendly, and inserting!
iq = """INSERT INTO epa_pollution (""" + ','.join(merged_epa.columns) + """) VALUES (%s, %s, %s, %s, %s, %s, %s);"""
cur.executemany(iq, epa_tups)
conn.commit()

# Preview
sql_head('epa_pollution')

,countyfips,ozone_concentration,ozone_aqi,no2_concentration,no2_aqi,pm25_concentration,pm25_aqi
0,1003,0.039370,37.049380,None,None,7.317391,38.269566
1,1027,NaN,NaN,None,None,6.880869,35.921738
2,1049,0.038920,36.366390,None,None,6.855932,36.135593
3,1051,0.036412,33.936974,None,None,NaN,NaN
4,1055,0.040882,38.457985,None,None,8.212079,41.415730


## Loading in nhts_travel
This table will have information on outbound/inbound trips with percents of modes/purpose/distance category for each unique CBSA code!

In [ ]:
merged_nhts_data = merged_nhts_data.fillna(np.nan).replace([np.nan], [None]) # making it sql friendly
merged_nhts_data.dtypes

,0
cbsacode,int64
outbound_annual_trips,int64
outbound_mode_air_pct,float64
outbound_mode_rail_pct,float64
outbound_mode_vehicle_pct,float64
outbound_mode_atf_pct,float64
outbound_purpose_work_pct,float64
outbound_purpose_nonwork_pct,float64
outbound_distance_low_pct,float64
outbound_distance_high_pct,float64


In [ ]:
my_drop_table('nhts_travel') # dropping if already exists

conn, cur = get_conn_cur()
tq = """
CREATE TABLE nhts_travel (
    cbsacode INTEGER PRIMARY KEY,
    outbound_annual_trips BIGINT,
    outbound_mode_air_pct FLOAT,
    outbound_mode_rail_pct FLOAT,
    outbound_mode_vehicle_pct FLOAT,
    outbound_mode_atf_pct FLOAT,
    outbound_purpose_work_pct FLOAT,
    outbound_purpose_nonwork_pct FLOAT,
    outbound_distance_low_pct FLOAT,
    outbound_distance_high_pct FLOAT,
    inbound_annual_trips BIGINT,
    inbound_mode_air_pct FLOAT,
    inbound_mode_rail_pct FLOAT,
    inbound_mode_vehicle_pct FLOAT,
    inbound_mode_atf_pct FLOAT,
    inbound_purpose_work_pct FLOAT,
    inbound_purpose_nonwork_pct FLOAT,
    inbound_distance_low_pct FLOAT,
    inbound_distance_high_pct FLOAT
);
"""
cur.execute(tq)
conn.commit()
nhts_tups = [tuple(x.item() if hasattr(x, 'item') else x for x in row) for row in merged_nhts_data.to_numpy()] # converting rows to tuples, making it python friendly, and inserting!
placeholders = ','.join(['%s'] * len(merged_nhts_data.columns))
iq = f"""INSERT INTO nhts_travel ({','.join(merged_nhts_data.columns)}) VALUES ({placeholders});"""
cur.executemany(iq, nhts_tups)
conn.commit()
sql_head('nhts_travel')

,cbsacode,outbound_annual_trips,outbound_mode_air_pct,outbound_mode_rail_pct,outbound_mode_vehicle_pct,outbound_mode_atf_pct,outbound_purpose_work_pct,outbound_purpose_nonwork_pct,outbound_distance_low_pct,outbound_distance_high_pct,inbound_annual_trips,inbound_mode_air_pct,inbound_mode_rail_pct,inbound_mode_vehicle_pct,inbound_mode_atf_pct,inbound_purpose_work_pct,inbound_purpose_nonwork_pct,inbound_distance_low_pct,inbound_distance_high_pct
0,10180,182042332,0.000826,0.000000,0.912730,0.086444,0.306203,0.693797,0.982089,0.017911,181977456,0.001134,0.000000,0.912390,0.086475,0.306048,0.693952,0.982438,0.017562
1,10420,702867310,0.000555,0.000001,0.877631,0.121812,0.280283,0.719717,0.992195,0.007805,700001806,0.000640,0.000001,0.877162,0.122196,0.280171,0.719829,0.992352,0.007648
2,10500,163588267,0.000518,0.000000,0.900179,0.099303,0.309609,0.690391,0.986906,0.013094,163689800,0.000450,0.000000,0.900312,0.099238,0.309431,0.690569,0.986270,0.013730
3,10540,121896038,0.000000,0.000242,0.840219,0.159539,0.292488,0.707512,0.969421,0.030579,121727083,0.000000,0.000254,0.840233,0.159513,0.291599,0.708401,0.971937,0.028063
4,10580,870348816,0.001720,0.000632,0.779819,0.217829,0.287608,0.712392,0.981961,0.018039,873275147,0.001828,0.000789,0.779664,0.217719,0.287233,0.712767,0.981542,0.018458


# Example Queries

Now that everything is loaded in and it works, let's try out some test queries!

The initial idea was to be able to query by FIPS or cbsa, so let's try out those, and also see what happens if we query by a specific state/value.

In [ ]:
# QUERYING BY SPECIFIC CBSA:
cur = conn.cursor()

query = """
SELECT
    county_info.CountyFIPS,
    county_info.state,
    county_info.county,
    county_info.cbsacode,
    cdc_outcomes.copd_adj_prev,
    epa_pollution.ozone_concentration

FROM county_info

LEFT JOIN cdc_outcomes
    ON county_info.CountyFIPS = cdc_outcomes.CountyFIPS

LEFT JOIN epa_pollution
    ON county_info.CountyFIPS = epa_pollution.CountyFIPS

WHERE county_info.cbsacode = 33860;
"""

cur.execute(query)
rows = cur.fetchall()
for row in rows:
    print(row)

(1051, 'Alabama', 'Elmore', 33860, 6.8079713400298285, 0.036411766)
(1101, 'Alabama', 'Montgomery', 33860, 6.860906183403499, 0.041812226)
(1085, 'Alabama', 'Lowndes', 33860, 10.243690479688965, None)
(1001, 'Alabama', 'Autauga', 33860, 6.408057565265498, None)


In [ ]:
# QUERYING TO LOOK FOR ALL COUNTYFIPS THAT MEET SPECIFIC REQUIREMENTS (adjusted prevalence of asthma being greater than 3.0 and concentration being greater than 5)
cur = conn.cursor()

query = """
SELECT
    county_info.CountyFIPS,
    county_info.state,
    county_info.county,
    cdc_outcomes.asthma_adj_prev,
    epa_pollution.pm25_concentration

FROM county_info

JOIN cdc_outcomes
    ON county_info.CountyFIPS = cdc_outcomes.CountyFIPS

JOIN epa_pollution
    ON county_info.CountyFIPS = epa_pollution.CountyFIPS

WHERE
    cdc_outcomes.asthma_adj_prev > 3
    AND epa_pollution.pm25_concentration > 5;
"""

cur.execute(query)
rows = cur.fetchall()
for row in rows:
    print(row)

(1003, 'Alabama', 'Baldwin', 7.981697345678039, 7.3173914)
(1027, 'Alabama', 'Clay', 8.563743947572458, 6.8808694)
(1049, 'Alabama', 'DeKalb', 8.421757514511825, 6.855932)
(1055, 'Alabama', 'Etowah', 8.75607083606369, 8.212079)
(1073, 'Alabama', 'Jefferson', 8.59227168318914, 8.797080771428572)
(1079, 'Alabama', 'Lawrence', 8.713677558563127, 5.527451)
(1089, 'Alabama', 'Madison', 8.055033396758823, 7.6835613)
(1097, 'Alabama', 'Mobile', 8.478681165996486, 7.7818794)
(1101, 'Alabama', 'Montgomery', 8.63696019037492, 6.805812)
(1103, 'Alabama', 'Morgan', 8.095695750658656, 8.322118)
(1113, 'Alabama', 'Russell', 8.791202996458345, 9.007022)
(1119, 'Alabama', 'Sumter', 9.70892171302663, 6.836963)
(1125, 'Alabama', 'Tuscaloosa', 8.810590807966767, 7.7115936)
(2090, 'Alaska', 'Fairbanks North Star', 7.973906424950247, 12.269479833333333)
(2110, 'Alaska', 'Juneau', 8.082335780563723, 5.063912)
(4013, 'Arizona', 'Maricopa', 7.989914596769591, 8.07820531)
(4021, 'Arizona', 'Pinal', 8.542072009

In [ ]:
cur = conn.cursor()

# I'm going to query for a specific CountyFIPs, and then do a left join to get data from cdc_outcomes and epa_pollution!
query = """
SELECT
    county_info.state,
    county_info.county,
    cdc_outcomes.asthma_crude_prev,
    epa_pollution.pm25_concentration

FROM county_info

LEFT JOIN cdc_outcomes
    ON county_info.CountyFIPS = cdc_outcomes.CountyFIPS

LEFT JOIN epa_pollution
    ON county_info.CountyFIPS = epa_pollution.CountyFIPS

WHERE county_info.CountyFIPS = 1001;
"""

cur.execute(query)
rows = cur.fetchall()
for row in rows:
    print(row)


('Alabama', 'Autauga', 10.447058823529412, None)
